When Does Trajectory Context Improve Failure Prediction?

In [3]:
# ============================================================
# 1. Imports
# ============================================================

import numpy as np
import pandas as pd

from sentence_transformers import SentenceTransformer

from sklearn.model_selection import StratifiedGroupKFold
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    classification_report,
    average_precision_score,
    roc_auc_score,
)

RANDOM_STATE = 42

In [2]:
# ============================================================
# 2. Load canonical trajectory exports
# ============================================================

train_events = pd.read_csv(
    "../data/processed/trajectory_events_train.csv"
)

test_events = pd.read_csv(
    "../data/processed/trajectory_events_test.csv"
)

targets = pd.read_csv(
    "../data/processed/trajectory_targets.csv"
)

train_targets = (
    targets[
        targets["split"] == "train"
    ]
    .reset_index(drop=True)
)

test_targets = (
    targets[
        targets["split"] == "test"
    ]
    .reset_index(drop=True)
)

print("Train events:", train_events.shape)
print("Test events:", test_events.shape)

print("Train targets:", train_targets.shape)
print("Test targets:", test_targets.shape)

assert len(train_targets) == 1489
assert len(test_targets) == 287

Train events: (3792, 40)
Test events: (799, 40)
Train targets: (1489, 14)
Test targets: (287, 14)


In [4]:
# ============================================================
# 3. Labels and groups
# ============================================================

LABEL_NAMES = [
    "workflow_error",
    "constraint_error",
    "tool_use_error",
    "grounding_state_error",
    "reasoning_value_error",
]

y_train = (
    train_targets[
        "family_label"
    ]
    .to_numpy()
    .astype(int)
)

y_test = (
    test_targets[
        "family_label"
    ]
    .to_numpy()
    .astype(int)
)

group_col = (
    "canonical_group"
    if "canonical_group"
    in train_targets.columns
    else "group_id"
)

groups_train = (
    train_targets[
        group_col
    ]
    .astype(str)
    .to_numpy()
)

groups_test = (
    test_targets[
        group_col
    ]
    .astype(str)
    .to_numpy()
)

print("Group column:", group_col)
print("Train groups:", len(np.unique(groups_train)))
print("Test groups:", len(np.unique(groups_test)))

assert len(y_train) == 1489
assert len(groups_train) == 1489

Group column: canonical_group
Train groups: 335
Test groups: 84


In [5]:
# ============================================================
# 4. Reconstruct target histories
# ============================================================

TRAJECTORY_KEY = [
    "dataset",
    "group_id",
]


def build_history_indices(
    events_df,
    targets_df,
):

    trajectory_events = {}

    for key, group in events_df.groupby(
        TRAJECTORY_KEY,
        sort=False,
    ):

        group = group.sort_values(
            "message_index"
        )

        trajectory_events[key] = group

    histories = []

    for _, row in targets_df.iterrows():

        key = (
            row["dataset"],
            row["group_id"],
        )

        target_message_index = int(
            row["message_index"]
        )

        events = trajectory_events.get(key)

        if events is None:
            histories.append([])
            continue

        indices = (
            events.loc[
                events["message_index"]
                < target_message_index
            ]
            .index
            .tolist()
        )

        histories.append(indices)

    return histories


train_history_indices = build_history_indices(
    train_events,
    train_targets,
)

test_history_indices = build_history_indices(
    test_events,
    test_targets,
)

In [6]:
# ============================================================
# 5. Verify exact historical alignment
# ============================================================

train_history_lengths = np.array([
    len(x)
    for x in train_history_indices
])

test_history_lengths = np.array([
    len(x)
    for x in test_history_indices
])


np.testing.assert_array_equal(
    train_history_lengths,
    train_targets[
        "history_event_count"
    ].to_numpy(),
)

np.testing.assert_array_equal(
    test_history_lengths,
    test_targets[
        "history_event_count"
    ].to_numpy(),
)

print("✓ History reconstruction exact")

✓ History reconstruction exact


In [7]:
# ============================================================
# 6. Clean event text
# ============================================================

def make_event_text(row):

    role = str(
        row["event_role"]
    ).strip()

    content = (
        ""
        if pd.isna(row["content"])
        else str(row["content"]).strip()
    )

    prefix = f"[{role}]"

    if content.upper().startswith(
        prefix.upper()
    ):
        return content

    return (
        prefix
        + "\n"
        + content
    )


train_events["event_text"] = (
    train_events.apply(
        make_event_text,
        axis=1,
    )
)

test_events["event_text"] = (
    test_events.apply(
        make_event_text,
        axis=1,
    )
)

In [8]:
# ============================================================
# 7. Semantic encoder
# ============================================================

semantic_model = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2"
)

In [9]:
# ============================================================
# 8. Embed current targets
# ============================================================

train_current_embeddings = (
    semantic_model.encode(
        train_targets[
            "content"
        ]
        .fillna("")
        .astype(str)
        .tolist(),
        batch_size=64,
        show_progress_bar=True,
        normalize_embeddings=True,
    )
)

test_current_embeddings = (
    semantic_model.encode(
        test_targets[
            "content"
        ]
        .fillna("")
        .astype(str)
        .tolist(),
        batch_size=64,
        show_progress_bar=True,
        normalize_embeddings=True,
    )
)

print(
    train_current_embeddings.shape,
    test_current_embeddings.shape,
)

Batches:   0%|          | 0/24 [00:00<?, ?it/s]

Batches:   0%|          | 0/5 [00:00<?, ?it/s]

(1489, 384) (287, 384)


In [10]:
# ============================================================
# 9. Embed historical events
# ============================================================

train_event_embeddings = (
    semantic_model.encode(
        train_events[
            "event_text"
        ].tolist(),
        batch_size=64,
        show_progress_bar=True,
        normalize_embeddings=True,
    )
)

test_event_embeddings = (
    semantic_model.encode(
        test_events[
            "event_text"
        ].tolist(),
        batch_size=64,
        show_progress_bar=True,
        normalize_embeddings=True,
    )
)

print(
    train_event_embeddings.shape,
    test_event_embeddings.shape,
)

Batches:   0%|          | 0/60 [00:00<?, ?it/s]

Batches:   0%|          | 0/13 [00:00<?, ?it/s]

(3792, 384) (799, 384)


In [11]:
# ============================================================
# 10. Fixed last-three historical events
# ============================================================

def get_last_k_event_embeddings(
    history_indices,
    event_embeddings,
    k=3,
):

    n = len(history_indices)
    d = event_embeddings.shape[1]

    output = np.zeros(
        (n, k, d),
        dtype=np.float32,
    )

    for i, indices in enumerate(
        history_indices
    ):

        if not indices:
            continue

        recent = indices[-k:]

        output[
            i,
            -len(recent):,
            :
        ] = event_embeddings[
            recent
        ]

    return output


H_train_3 = get_last_k_event_embeddings(
    train_history_indices,
    train_event_embeddings,
    k=3,
)

H_test_3 = get_last_k_event_embeddings(
    test_history_indices,
    test_event_embeddings,
    k=3,
)

print(
    H_train_3.shape,
    H_test_3.shape,
)

(1489, 3, 384) (287, 3, 384)


In [12]:
# ============================================================
# 11. Compact local transition features
# ============================================================

def cosine_rows(a, b):

    numerator = np.sum(
        a * b,
        axis=1,
    )

    denominator = (
        np.linalg.norm(
            a,
            axis=1,
        )
        *
        np.linalg.norm(
            b,
            axis=1,
        )
        + 1e-8
    )

    return numerator / denominator


def build_transition_features(
    current,
    history,
):

    columns = []
    names = []

    k = history.shape[1]

    for pos in range(k):

        h = history[:, pos, :]

        present = (
            np.linalg.norm(
                h,
                axis=1,
            ) > 1e-8
        ).astype(float)

        cosine = (
            cosine_rows(
                current,
                h,
            )
            * present
        )

        l1 = (
            np.mean(
                np.abs(
                    current - h
                ),
                axis=1,
            )
            * present
        )

        l2 = (
            np.linalg.norm(
                current - h,
                axis=1,
            )
            * present
        )

        relative_pos = (
            k - pos
        )

        columns.extend([
            cosine,
            l1,
            l2,
            present,
        ])

        names.extend([
            f"current_tminus{relative_pos}_cosine",
            f"current_tminus{relative_pos}_l1",
            f"current_tminus{relative_pos}_l2",
            f"tminus{relative_pos}_present",
        ])

    # history-to-history transitions
    for pos in range(1, k):

        older = history[
            :,
            pos - 1,
            :
        ]

        newer = history[
            :,
            pos,
            :
        ]

        pair_present = (
            (
                np.linalg.norm(
                    older,
                    axis=1,
                ) > 1e-8
            )
            &
            (
                np.linalg.norm(
                    newer,
                    axis=1,
                ) > 1e-8
            )
        ).astype(float)

        cosine = (
            cosine_rows(
                older,
                newer,
            )
            * pair_present
        )

        l2 = (
            np.linalg.norm(
                newer - older,
                axis=1,
            )
            * pair_present
        )

        columns.extend([
            cosine,
            l2,
        ])

        names.extend([
            f"history_transition_{pos}_cosine",
            f"history_transition_{pos}_l2",
        ])

    return (
        np.column_stack(
            columns
        ).astype(np.float32),
        names,
    )


T_train, transition_names = (
    build_transition_features(
        train_current_embeddings,
        H_train_3,
    )
)

T_test, _ = (
    build_transition_features(
        test_current_embeddings,
        H_test_3,
    )
)

print("Transition train:", T_train.shape)
print("Transition test:", T_test.shape)

print("\nFeatures:")
for x in transition_names:
    print(" ", x)

Transition train: (1489, 16)
Transition test: (287, 16)

Features:
  current_tminus3_cosine
  current_tminus3_l1
  current_tminus3_l2
  tminus3_present
  current_tminus2_cosine
  current_tminus2_l1
  current_tminus2_l2
  tminus2_present
  current_tminus1_cosine
  current_tminus1_l1
  current_tminus1_l2
  tminus1_present
  history_transition_1_cosine
  history_transition_1_l2
  history_transition_2_cosine
  history_transition_2_l2


In [13]:
# ============================================================
# 12. Expert feature spaces
# ============================================================

X_sem_train = np.asarray(
    train_current_embeddings,
    dtype=np.float32,
)

X_sem_test = np.asarray(
    test_current_embeddings,
    dtype=np.float32,
)


X_trans_train = np.hstack([
    X_sem_train,
    T_train,
])

X_trans_test = np.hstack([
    X_sem_test,
    T_test,
])


print(
    "Semantic:",
    X_sem_train.shape,
    X_sem_test.shape,
)

print(
    "Transition:",
    X_trans_train.shape,
    X_trans_test.shape,
)

Semantic: (1489, 384) (287, 384)
Transition: (1489, 400) (287, 400)


In [14]:
# ============================================================
# 13. Group-safe OOF expert predictions
# ============================================================

N_CLASSES = 5

oof_sem_prob = np.zeros(
    (len(y_train), N_CLASSES),
    dtype=np.float32,
)

oof_trans_prob = np.zeros(
    (len(y_train), N_CLASSES),
    dtype=np.float32,
)


cv = StratifiedGroupKFold(
    n_splits=5,
    shuffle=True,
    random_state=RANDOM_STATE,
)


for fold, (
    tr_idx,
    va_idx,
) in enumerate(
    cv.split(
        X_sem_train,
        y_train,
        groups=groups_train,
    ),
    start=1,
):

    semantic_model_fold = make_pipeline(
        StandardScaler(),

        LogisticRegression(
            C=0.03,
            max_iter=5000,
            random_state=RANDOM_STATE,
        ),
    )

    transition_model_fold = make_pipeline(
        StandardScaler(),

        LogisticRegression(
            C=0.01,
            max_iter=5000,
            random_state=RANDOM_STATE,
        ),
    )

    semantic_model_fold.fit(
        X_sem_train[tr_idx],
        y_train[tr_idx],
    )

    transition_model_fold.fit(
        X_trans_train[tr_idx],
        y_train[tr_idx],
    )

    oof_sem_prob[va_idx] = (
        semantic_model_fold.predict_proba(
            X_sem_train[va_idx]
        )
    )

    oof_trans_prob[va_idx] = (
        transition_model_fold.predict_proba(
            X_trans_train[va_idx]
        )
    )

    print(
        f"Fold {fold} complete"
    )

Fold 1 complete
Fold 2 complete
Fold 3 complete
Fold 4 complete
Fold 5 complete


In [15]:
# ============================================================
# 14. Expert outcome categories
# ============================================================

oof_sem_pred = (
    oof_sem_prob.argmax(
        axis=1
    )
)

oof_trans_pred = (
    oof_trans_prob.argmax(
        axis=1
    )
)


sem_correct = (
    oof_sem_pred
    == y_train
)

trans_correct = (
    oof_trans_pred
    == y_train
)


rescue = (
    (~sem_correct)
    &
    trans_correct
)

break_case = (
    sem_correct
    &
    (~trans_correct)
)

both_correct = (
    sem_correct
    &
    trans_correct
)

both_wrong = (
    (~sem_correct)
    &
    (~trans_correct)
)


outcome = np.select(
    [
        rescue,
        break_case,
        both_correct,
    ],
    [
        "rescue",
        "break",
        "both_correct",
    ],
    default="both_wrong",
)


print(
    pd.Series(
        outcome
    ).value_counts()
)

both_correct    728
both_wrong      638
rescue           69
break            54
Name: count, dtype: int64


In [16]:
# ============================================================
# 15. Utility-analysis dataframe
# ============================================================

utility_df = pd.DataFrame(
    T_train,
    columns=transition_names,
)

utility_df[
    "failure_family"
] = (
    train_targets[
        "failure_family"
    ].to_numpy()
)

utility_df[
    "current_role"
] = (
    train_targets[
        "event_role"
    ].to_numpy()
    if "event_role"
    in train_targets.columns
    else "UNKNOWN"
)

utility_df[
    "history_event_count"
] = (
    train_targets[
        "history_event_count"
    ].to_numpy()
)

utility_df[
    "semantic_pred"
] = (
    oof_sem_pred
)

utility_df[
    "transition_pred"
] = (
    oof_trans_pred
)

utility_df[
    "semantic_correct"
] = (
    sem_correct
)

utility_df[
    "transition_correct"
] = (
    trans_correct
)

utility_df[
    "outcome"
] = (
    outcome
)

In [17]:
# ============================================================
# 16. Confidence / uncertainty features
# ============================================================

def probability_meta(prob):

    sorted_prob = np.sort(
        prob,
        axis=1,
    )

    confidence = (
        sorted_prob[:, -1]
    )

    margin = (
        sorted_prob[:, -1]
        -
        sorted_prob[:, -2]
    )

    entropy = -np.sum(
        prob
        * np.log(
            prob + 1e-8
        ),
        axis=1,
    )

    return (
        confidence,
        margin,
        entropy,
    )


(
    sem_confidence,
    sem_margin,
    sem_entropy,
) = probability_meta(
    oof_sem_prob
)


(
    trans_confidence,
    trans_margin,
    trans_entropy,
) = probability_meta(
    oof_trans_prob
)


utility_df[
    "semantic_confidence"
] = sem_confidence

utility_df[
    "semantic_margin"
] = sem_margin

utility_df[
    "semantic_entropy"
] = sem_entropy

utility_df[
    "transition_confidence"
] = trans_confidence

utility_df[
    "transition_margin"
] = trans_margin

utility_df[
    "transition_entropy"
] = trans_entropy


utility_df[
    "confidence_delta"
] = (
    trans_confidence
    -
    sem_confidence
)

utility_df[
    "experts_disagree"
] = (
    oof_sem_pred
    != oof_trans_pred
).astype(int)

In [18]:
# ============================================================
# 17. Main descriptive outcome comparison
# ============================================================

analysis_features = [
    "semantic_confidence",
    "semantic_margin",
    "semantic_entropy",

    "transition_confidence",
    "transition_margin",
    "transition_entropy",

    "confidence_delta",

    "current_tminus3_cosine",
    "current_tminus2_cosine",
    "current_tminus1_cosine",

    "history_transition_1_cosine",
    "history_transition_2_cosine",

    "current_tminus3_l2",
    "current_tminus2_l2",
    "current_tminus1_l2",

    "history_event_count",
]


outcome_summary = (
    utility_df
    .groupby(
        "outcome"
    )[analysis_features]
    .agg([
        "count",
        "mean",
        "median",
        "std",
    ])
)


display(
    outcome_summary.round(4)
)

semantic_confidence                         semantic_margin  \
                           count    mean  median     std           count   
outcome                                                                    
both_correct                 728  0.7536  0.8099  0.1791             728   
both_wrong                   638  0.6317  0.6240  0.1642             638   
break                         54  0.4819  0.4868  0.0856              54   
rescue                        69  0.4678  0.4666  0.0862              69   

                                     semantic_entropy          ...  \
                mean  median     std            count    mean  ...   
outcome                                                        ...   
both_correct  0.5934  0.6741  0.3006              728  0.6654  ...   
both_wrong    0.4006  0.3742  0.2611              638  0.9030  ...   
break         0.1426  0.1104  0.1207               54  1.0913  ...   
rescue        0.1144  0.0742  0.1223               69  1.0899  ...   

             current_tminus2_l2         current_tminus1_l2                  \
                         median     std              count    mean  median   
outcome                                                                      
both_correct             0.8330  0.4572                728  0.7942  0.8320   
both_wrong               0.9781  0.3802                638  0.9089  0.9810   
break                    0.9416  0.4237                 54  0.8786  0.9638   
rescue                   0.8673  0.3993                 69  0.8897  0.9815   

                     history_event_count                           
                 std               count     mean median      std  
outcome                                                            
both_correct  0.4050                 728  19.7665   11.0  20.6746  
both_wrong    0.3260                 638  12.5219    8.0  15.2495  
break         0.3891                  54   9.4259    7.0   9.7567  
rescue        0.3337                  69   8.3188    8.0   5.1834  

[4 rows x 64 columns]

In [19]:
# ============================================================
# 18. Rescue vs break comparison
# ============================================================

rescue_break_df = (
    utility_df[
        utility_df[
            "outcome"
        ].isin([
            "rescue",
            "break",
        ])
    ]
)


rescue_break_summary = (
    rescue_break_df
    .groupby(
        "outcome"
    )[analysis_features]
    .mean()
    .T
)


rescue_break_summary[
    "difference_rescue_minus_break"
] = (
    rescue_break_summary[
        "rescue"
    ]
    -
    rescue_break_summary[
        "break"
    ]
)


display(
    rescue_break_summary
    .sort_values(
        "difference_rescue_minus_break",
        ascending=False,
    )
    .round(4)
)

outcome,break,rescue,difference_rescue_minus_break
current_tminus2_cosine,0.4153,0.5254,0.1101
current_tminus1_cosine,0.4471,0.5204,0.0732
current_tminus3_l2,0.8506,0.9136,0.0630
current_tminus3_cosine,0.3545,0.3853,0.0309
current_tminus1_l2,0.8786,0.8897,0.0111
confidence_delta,-0.0355,-0.0267,0.0088
transition_entropy,1.1698,1.1781,0.0083
history_transition_2_cosine,0.4880,0.4959,0.0079
history_transition_1_cosine,0.4678,0.4735,0.0057
semantic_entropy,1.0913,1.0899,-0.0014


In [20]:
# ============================================================
# 19. Utility by failure family
# ============================================================

class_utility = (
    utility_df
    .groupby(
        "failure_family"
    )
    .agg(
        count=(
            "outcome",
            "size",
        ),

        rescues=(
            "outcome",
            lambda x:
                (x == "rescue").sum(),
        ),

        breaks=(
            "outcome",
            lambda x:
                (x == "break").sum(),
        ),

        semantic_correct=(
            "semantic_correct",
            "sum",
        ),

        transition_correct=(
            "transition_correct",
            "sum",
        ),
    )
)


class_utility[
    "net_rescues"
] = (
    class_utility[
        "rescues"
    ]
    -
    class_utility[
        "breaks"
    ]
)


class_utility[
    "rescue_rate"
] = (
    class_utility[
        "rescues"
    ]
    /
    class_utility[
        "count"
    ]
)


class_utility[
    "break_rate"
] = (
    class_utility[
        "breaks"
    ]
    /
    class_utility[
        "count"
    ]
)


display(
    class_utility
    .sort_values(
        "net_rescues",
        ascending=False,
    )
    .round(4)
)

,count,rescues,breaks,semantic_correct,transition_correct,net_rescues,rescue_rate,break_rate
failure_family,,,,,,,,
constraint_error,317,26,7,121,140,19,0.0820,0.0221
workflow_error,660,28,20,426,434,8,0.0424,0.0303
reasoning_value_error,31,0,2,16,14,-2,0.0000,0.0645
grounding_state_error,244,12,15,112,109,-3,0.0492,0.0615
tool_use_error,237,3,10,107,100,-7,0.0127,0.0422


In [21]:
# ============================================================
# 20. Disagreement-only utility dataset
# ============================================================

disagreement_mask = (
    oof_sem_pred
    != oof_trans_pred
)

utility_disagree = (
    utility_df.loc[
        disagreement_mask
    ]
    .copy()
)

utility_disagree[
    "trust_transition"
] = (
    trans_correct[
        disagreement_mask
    ]
    &
    (~sem_correct[
        disagreement_mask
    ])
).astype(int)


print(
    "Disagreement rows:",
    len(utility_disagree),
)

print(
    "\nRouting target:"
)

print(
    utility_disagree[
        "trust_transition"
    ].value_counts()
)

print(
    "\nTransition share:",
    utility_disagree[
        "trust_transition"
    ].mean()
)

Disagreement rows: 183

Routing target:
trust_transition
0    114
1     69
Name: count, dtype: int64

Transition share: 0.3770491803278688


In [23]:
# ============================================================
# 21. Predicted class features
# ============================================================

utility_disagree[
    "semantic_pred_name"
] = [
    LABEL_NAMES[int(x)]
    for x in utility_disagree[
        "semantic_pred"
    ]
]

utility_disagree[
    "transition_pred_name"
] = [
    LABEL_NAMES[int(x)]
    for x in utility_disagree[
        "transition_pred"
    ]
]


display(
    pd.crosstab(
        utility_disagree[
            "semantic_pred_name"
        ],
        utility_disagree[
            "transition_pred_name"
        ],
    )
)

transition_pred_name,constraint_error,grounding_state_error,reasoning_value_error,tool_use_error,workflow_error
semantic_pred_name,,,,,
constraint_error,0,10,0,3,18
grounding_state_error,17,0,2,7,23
reasoning_value_error,0,2,0,2,1
tool_use_error,11,5,0,0,31
workflow_error,28,13,0,10,0


In [24]:
# ============================================================
# 22. Routing success by expert disagreement type
# ============================================================

pair_utility = (
    utility_disagree
    .groupby([
        "semantic_pred_name",
        "transition_pred_name",
    ])
    .agg(
        count=(
            "trust_transition",
            "size",
        ),

        transition_wins=(
            "trust_transition",
            "sum",
        ),
    )
)

pair_utility[
    "transition_win_rate"
] = (
    pair_utility[
        "transition_wins"
    ]
    /
    pair_utility[
        "count"
    ]
)

display(
    pair_utility
    .sort_values(
        [
            "count",
            "transition_win_rate",
        ],
        ascending=[
            False,
            False,
        ],
    )
    .round(4)
)

,,count,transition_wins,transition_win_rate
semantic_pred_name,transition_pred_name,,,
tool_use_error,workflow_error,31,15,0.4839
workflow_error,constraint_error,28,15,0.5357
grounding_state_error,workflow_error,23,5,0.2174
constraint_error,workflow_error,18,8,0.4444
grounding_state_error,constraint_error,17,4,0.2353
workflow_error,grounding_state_error,13,4,0.3077
tool_use_error,constraint_error,11,7,0.6364
constraint_error,grounding_state_error,10,5,0.5000
workflow_error,tool_use_error,10,2,0.2000


In [25]:
# ============================================================
# 23. Compact interpretable router features
# ============================================================

router_numeric_features = [
    "current_tminus1_cosine",
    "current_tminus2_cosine",
    "current_tminus3_cosine",

    "history_transition_1_cosine",
    "history_transition_2_cosine",

    "semantic_confidence",
    "semantic_margin",

    "transition_confidence",
    "transition_margin",

    "history_event_count",
]

In [26]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import (
    StandardScaler,
    OneHotEncoder,
)
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression


router_categorical_features = [
    "semantic_pred_name",
    "transition_pred_name",
]


router_preprocessor = ColumnTransformer(
    transformers=[
        (
            "num",
            StandardScaler(),
            router_numeric_features,
        ),
        (
            "cat",
            OneHotEncoder(
                handle_unknown="ignore"
            ),
            router_categorical_features,
        ),
    ]
)

In [27]:
# ============================================================
# 24. Interpretable disagreement router
# ============================================================

router_X_df = (
    utility_disagree[
        router_numeric_features
        + router_categorical_features
    ]
    .copy()
)

router_y = (
    utility_disagree[
        "trust_transition"
    ]
    .to_numpy()
)

router_groups = (
    groups_train[
        disagreement_mask
    ]
)


router_C_values = [
    0.01,
    0.03,
    0.1,
    0.3,
    1.0,
]

router_rows = []


router_cv = StratifiedGroupKFold(
    n_splits=5,
    shuffle=True,
    random_state=42,
)


for C in router_C_values:

    fold_scores = []

    for tr_idx, va_idx in router_cv.split(
        router_X_df,
        router_y,
        groups=router_groups,
    ):

        model = Pipeline([
            (
                "preprocessor",
                router_preprocessor,
            ),
            (
                "classifier",
                LogisticRegression(
                    C=C,
                    max_iter=5000,
                    random_state=42,
                ),
            ),
        ])

        model.fit(
            router_X_df.iloc[
                tr_idx
            ],
            router_y[
                tr_idx
            ],
        )

        pred = model.predict(
            router_X_df.iloc[
                va_idx
            ]
        )

        fold_scores.append({
            "accuracy":
                accuracy_score(
                    router_y[
                        va_idx
                    ],
                    pred,
                ),

            "balanced_accuracy":
                balanced_accuracy_score(
                    router_y[
                        va_idx
                    ],
                    pred,
                ),

            "macro_f1":
                f1_score(
                    router_y[
                        va_idx
                    ],
                    pred,
                    average="macro",
                    zero_division=0,
                ),
        })

    fold_df = pd.DataFrame(
        fold_scores
    )

    router_rows.append({
        "C": C,

        "accuracy":
            fold_df[
                "accuracy"
            ].mean(),

        "balanced_accuracy":
            fold_df[
                "balanced_accuracy"
            ].mean(),

        "macro_f1":
            fold_df[
                "macro_f1"
            ].mean(),

        "macro_f1_std":
            fold_df[
                "macro_f1"
            ].std(),
    })


interpretable_router_results = (
    pd.DataFrame(
        router_rows
    )
)


display(
    interpretable_router_results
    .sort_values(
        "macro_f1",
        ascending=False,
    )
    .round(4)
)

,C,accuracy,balanced_accuracy,macro_f1,macro_f1_std
4,1.00,0.6331,0.5963,0.5972,0.0794
3,0.30,0.6221,0.5680,0.5656,0.0749
2,0.10,0.5950,0.5203,0.4994,0.1049
1,0.03,0.6068,0.4934,0.4022,0.0774
0,0.01,0.6229,0.5000,0.3838,0.0047


In [28]:
# ============================================================
# 25. Cross-fitted compact disagreement router
# ============================================================

BEST_ROUTER_C = 1.0

compact_router_oof_choice = np.full(
    len(y_train),
    -1,
    dtype=int,
)

# Original positions of all expert disagreements
disagreement_indices = np.where(
    disagreement_mask
)[0]

router_X_all = (
    utility_disagree[
        router_numeric_features
        + router_categorical_features
    ]
    .reset_index(drop=True)
)

router_y_all = (
    utility_disagree[
        "trust_transition"
    ]
    .to_numpy()
    .astype(int)
)

router_groups_all = (
    groups_train[
        disagreement_mask
    ]
)

print("Router rows:", len(router_X_all))
print("Transition wins:", router_y_all.sum())
print("Semantic/default:", (router_y_all == 0).sum())

Router rows: 183
Transition wins: 69
Semantic/default: 114


In [29]:
# ============================================================
# 26. Group-safe compact-router OOF decisions
# ============================================================

compact_router_cv = StratifiedGroupKFold(
    n_splits=5,
    shuffle=True,
    random_state=42,
)

router_choice_disagree = np.full(
    len(router_y_all),
    -1,
    dtype=int,
)

router_prob_disagree = np.zeros(
    len(router_y_all),
    dtype=np.float32,
)


for fold, (tr_idx, va_idx) in enumerate(
    compact_router_cv.split(
        router_X_all,
        router_y_all,
        groups=router_groups_all,
    ),
    start=1,
):

    model = Pipeline([
        (
            "preprocessor",
            router_preprocessor,
        ),
        (
            "classifier",
            LogisticRegression(
                C=BEST_ROUTER_C,
                max_iter=5000,
                random_state=42,
            ),
        ),
    ])

    model.fit(
        router_X_all.iloc[tr_idx],
        router_y_all[tr_idx],
    )

    router_choice_disagree[va_idx] = (
        model.predict(
            router_X_all.iloc[va_idx]
        )
    )

    router_prob_disagree[va_idx] = (
        model.predict_proba(
            router_X_all.iloc[va_idx]
        )[:, 1]
    )

    print(f"Fold {fold} complete")


assert (
    router_choice_disagree >= 0
).all()

print("✓ Compact router cross-fitting complete")

Fold 1 complete
Fold 2 complete
Fold 3 complete
Fold 4 complete
Fold 5 complete
✓ Compact router cross-fitting complete


In [30]:
# ============================================================
# 27. End-to-end compact routed predictions
# ============================================================

compact_routed_pred = (
    oof_sem_pred.copy()
)

choose_transition = (
    router_choice_disagree == 1
)

transition_original_indices = (
    disagreement_indices[
        choose_transition
    ]
)

compact_routed_pred[
    transition_original_indices
] = (
    oof_trans_pred[
        transition_original_indices
    ]
)

print(
    "Expert disagreements:",
    len(disagreement_indices),
)

print(
    "Router chose transition:",
    choose_transition.sum(),
)

print(
    "Router chose semantic:",
    (~choose_transition).sum(),
)

Expert disagreements: 183
Router chose transition: 60
Router chose semantic: 123


In [31]:
# ============================================================
# 28. End-to-end comparison
# ============================================================

def evaluate_system(
    name,
    pred,
):

    return {
        "model": name,

        "accuracy":
            accuracy_score(
                y_train,
                pred,
            ),

        "balanced_accuracy":
            balanced_accuracy_score(
                y_train,
                pred,
            ),

        "macro_f1":
            f1_score(
                y_train,
                pred,
                average="macro",
                zero_division=0,
            ),

        "weighted_f1":
            f1_score(
                y_train,
                pred,
                average="weighted",
                zero_division=0,
            ),
    }


compact_routing_results = pd.DataFrame([
    evaluate_system(
        "semantic",
        oof_sem_pred,
    ),

    evaluate_system(
        "transition",
        oof_trans_pred,
    ),

    evaluate_system(
        "compact_interpretable_router",
        compact_routed_pred,
    ),
])

display(
    compact_routing_results
    .sort_values(
        "macro_f1",
        ascending=False,
    )
    .round(4)
)

,model,accuracy,balanced_accuracy,macro_f1,weighted_f1
2,compact_interpretable_router,0.5353,0.4918,0.5008,0.5335
1,transition,0.5353,0.4839,0.4981,0.5323
0,semantic,0.5252,0.4908,0.4974,0.5231


In [32]:
# ============================================================
# 29. Compact router rescue/break analysis
# ============================================================

semantic_correct_final = (
    oof_sem_pred == y_train
)

compact_correct = (
    compact_routed_pred == y_train
)

compact_rescues = (
    (~semantic_correct_final)
    &
    compact_correct
)

compact_breaks = (
    semantic_correct_final
    &
    (~compact_correct)
)


print(
    "Semantic correct:",
    semantic_correct_final.sum(),
)

print(
    "Compact routed correct:",
    compact_correct.sum(),
)

print(
    "Rescues:",
    compact_rescues.sum(),
)

print(
    "Breaks:",
    compact_breaks.sum(),
)

print(
    "Net rescues:",
    compact_rescues.sum()
    - compact_breaks.sum(),
)

print(
    "Accuracy delta:",
    compact_correct.mean()
    - semantic_correct_final.mean(),
)

Semantic correct: 782
Compact routed correct: 797
Rescues: 31
Breaks: 16
Net rescues: 15
Accuracy delta: 0.010073875083948969


In [33]:
# ============================================================
# 30. Fraction of oracle opportunity captured
# ============================================================

oracle_correct = (
    sem_correct
    | trans_correct
)

semantic_accuracy = (
    sem_correct.mean()
)

compact_accuracy = (
    compact_correct.mean()
)

oracle_accuracy = (
    oracle_correct.mean()
)


available_gain = (
    oracle_accuracy
    - semantic_accuracy
)

captured_gain = (
    compact_accuracy
    - semantic_accuracy
)

capture_fraction = (
    captured_gain
    / available_gain
    if available_gain > 0
    else np.nan
)


print(
    "Semantic accuracy:",
    semantic_accuracy,
)

print(
    "Compact router accuracy:",
    compact_accuracy,
)

print(
    "Oracle accuracy:",
    oracle_accuracy,
)

print(
    "Available oracle gain:",
    available_gain,
)

print(
    "Captured gain:",
    captured_gain,
)

print(
    "Fraction of oracle gain captured:",
    capture_fraction,
)

Semantic accuracy: 0.5251846877098724
Compact router accuracy: 0.5352585627938213
Oracle accuracy: 0.5715245130960376
Available oracle gain: 0.04633982538616521
Captured gain: 0.010073875083948969
Fraction of oracle gain captured: 0.2173913043478263


In [34]:
# ============================================================
# 31. Conditional utility analysis table
# ============================================================

conditional_df = utility_disagree.copy()

# Strong local continuity summary
conditional_df["recent_similarity_mean"] = (
    conditional_df[
        [
            "current_tminus1_cosine",
            "current_tminus2_cosine",
            "current_tminus3_cosine",
        ]
    ]
    .mean(axis=1)
)

conditional_df["recent_similarity_max"] = (
    conditional_df[
        [
            "current_tminus1_cosine",
            "current_tminus2_cosine",
            "current_tminus3_cosine",
        ]
    ]
    .max(axis=1)
)

conditional_df["history_similarity_mean"] = (
    conditional_df[
        [
            "history_transition_1_cosine",
            "history_transition_2_cosine",
        ]
    ]
    .mean(axis=1)
)

conditional_df[
    "semantic_uncertainty"
] = conditional_df[
    "semantic_entropy"
]

conditional_df[
    "semantic_low_margin"
] = (
    1.0
    - conditional_df[
        "semantic_margin"
    ]
)

print(
    conditional_df.shape
)

display(
    conditional_df[
        [
            "semantic_pred_name",
            "transition_pred_name",
            "trust_transition",
            "recent_similarity_mean",
            "semantic_confidence",
            "semantic_margin",
            "history_event_count",
        ]
    ].head()
)

(183, 40)


,semantic_pred_name,transition_pred_name,trust_transition,recent_similarity_mean,semantic_confidence,semantic_margin,history_event_count
5,workflow_error,constraint_error,1,0.000000,0.490638,0.134253,0
17,grounding_state_error,constraint_error,0,0.000000,0.637456,0.430462,0
21,workflow_error,grounding_state_error,1,0.462792,0.448689,0.029563,2
23,tool_use_error,grounding_state_error,0,0.769532,0.558958,0.188920,3
24,tool_use_error,grounding_state_error,0,0.675553,0.535681,0.197650,5


In [35]:
# ============================================================
# 32. Quantile bins
# ============================================================

def safe_qcut(series, q, labels):
    return pd.qcut(
        series,
        q=q,
        labels=labels,
        duplicates="drop",
    )


conditional_df["similarity_bin"] = safe_qcut(
    conditional_df[
        "recent_similarity_mean"
    ],
    q=3,
    labels=[
        "low",
        "medium",
        "high",
    ],
)

conditional_df["semantic_confidence_bin"] = safe_qcut(
    conditional_df[
        "semantic_confidence"
    ],
    q=3,
    labels=[
        "low",
        "medium",
        "high",
    ],
)

conditional_df["history_length_bin"] = pd.cut(
    conditional_df[
        "history_event_count"
    ],
    bins=[
        -1,
        3,
        10,
        np.inf,
    ],
    labels=[
        "short_0_3",
        "medium_4_10",
        "long_11_plus",
    ],
)

print(
    conditional_df[
        [
            "similarity_bin",
            "semantic_confidence_bin",
            "history_length_bin",
        ]
    ]
    .value_counts()
)

similarity_bin  semantic_confidence_bin  history_length_bin
high            high                     medium_4_10           15
                medium                   medium_4_10           14
low             low                      short_0_3             13
                high                     short_0_3             11
medium          medium                   medium_4_10           11
                low                      medium_4_10           10
high            medium                   long_11_plus           9
low             high                     medium_4_10            9
high            low                      long_11_plus           9
medium          low                      long_11_plus           9
low             medium                   medium_4_10            8
                low                      medium_4_10            8
medium          high                     long_11_plus           8
high            low                      medium_4_10            7
medium          

In [36]:
# ============================================================
# 33. Utility by recent semantic continuity
# ============================================================

similarity_utility = (
    conditional_df
    .groupby(
        "similarity_bin",
        observed=True,
    )
    .agg(
        count=(
            "trust_transition",
            "size",
        ),

        transition_wins=(
            "trust_transition",
            "sum",
        ),

        mean_similarity=(
            "recent_similarity_mean",
            "mean",
        ),
    )
)

similarity_utility[
    "transition_win_rate"
] = (
    similarity_utility[
        "transition_wins"
    ]
    /
    similarity_utility[
        "count"
    ]
)

display(
    similarity_utility.round(4)
)

,count,transition_wins,mean_similarity,transition_win_rate
similarity_bin,,,,
low,61,15,0.1825,0.2459
medium,61,33,0.4553,0.5410
high,61,21,0.7002,0.3443


In [37]:
# ============================================================
# 34. Similarity × semantic confidence interaction
# ============================================================

similarity_confidence = (
    conditional_df
    .groupby(
        [
            "similarity_bin",
            "semantic_confidence_bin",
        ],
        observed=True,
    )
    .agg(
        count=(
            "trust_transition",
            "size",
        ),

        transition_wins=(
            "trust_transition",
            "sum",
        ),
    )
)

similarity_confidence[
    "transition_win_rate"
] = (
    similarity_confidence[
        "transition_wins"
    ]
    /
    similarity_confidence[
        "count"
    ]
)

display(
    similarity_confidence
    .sort_values(
        "transition_win_rate",
        ascending=False,
    )
    .round(4)
)

count  transition_wins  \
similarity_bin semantic_confidence_bin                           
medium         low                         23               15   
               medium                      23               13   
high           high                        22               10   
low            medium                      14                5   
medium         high                        15                5   
high           medium                      23                7   
low            high                        24                7   
high           low                         16                4   
low            low                         23                3   

                                        transition_win_rate  
similarity_bin semantic_confidence_bin                       
medium         low                                   0.6522  
               medium                                0.5652  
high           high                                  0.4545  
low            medium                                0.3571  
medium         high                                  0.3333  
high           medium                                0.3043  
low            high                                  0.2917  
high           low                                   0.2500  
low            low                                   0.1304

In [38]:
# ============================================================
# 35. Prediction-pair × similarity interaction
# ============================================================

pair_similarity = (
    conditional_df
    .groupby(
        [
            "semantic_pred_name",
            "transition_pred_name",
            "similarity_bin",
        ],
        observed=True,
    )
    .agg(
        count=(
            "trust_transition",
            "size",
        ),

        transition_wins=(
            "trust_transition",
            "sum",
        ),

        mean_similarity=(
            "recent_similarity_mean",
            "mean",
        ),
    )
)

pair_similarity[
    "transition_win_rate"
] = (
    pair_similarity[
        "transition_wins"
    ]
    /
    pair_similarity[
        "count"
    ]
)

# Avoid overinterpreting cells with n=1.
pair_similarity_reliable = (
    pair_similarity[
        pair_similarity[
            "count"
        ] >= 5
    ]
)

display(
    pair_similarity_reliable
    .sort_values(
        [
            "transition_win_rate",
            "count",
        ],
        ascending=[
            False,
            False,
        ],
    )
    .round(4)
)

count  \
semantic_pred_name    transition_pred_name  similarity_bin          
workflow_error        constraint_error      medium             16   
tool_use_error        workflow_error        high               21   
                                            medium              6   
workflow_error        grounding_state_error medium              6   
constraint_error      grounding_state_error low                 7   
                      workflow_error        medium              7   
grounding_state_error constraint_error      medium              5   
constraint_error      workflow_error        high                8   
workflow_error        constraint_error      low                11   
grounding_state_error workflow_error        medium              6   
                                            high               17   
                      constraint_error      low                 6   
                                            high                6   
workflow_error        grounding_state_error low                 7   
                      tool_use_error        low                 6   
grounding_state_error tool_use_error        low                 5   

                                                            transition_wins  \
semantic_pred_name    transition_pred_name  similarity_bin                    
workflow_error        constraint_error      medium                       11   
tool_use_error        workflow_error        high                         11   
                                            medium                        3   
workflow_error        grounding_state_error medium                        3   
constraint_error      grounding_state_error low                           3   
                      workflow_error        medium                        3   
grounding_state_error constraint_error      medium                        2   
constraint_error      workflow_error        high                          3   
workflow_error        constraint_error      low                           4   
grounding_state_error workflow_error        medium                        2   
                                            high                          3   
                      constraint_error      low                           1   
                                            high                          1   
workflow_error        grounding_state_error low                           1   
                      tool_use_error        low                           0   
grounding_state_error tool_use_error        low                           0   

                                                            mean_similarity  \
semantic_pred_name    transition_pred_name  similarity_bin                    
workflow_error        constraint_error      medium                   0.4212   
tool_use_error        workflow_error        high                     0.7072   
                                            medium                   0.5078   
workflow_error        grounding_state_error medium                   0.4474   
constraint_error      grounding_state_error low                      0.1560   
                      workflow_error        medium                   0.4716   
grounding_state_error constraint_error      medium                   0.4755   
constraint_error      workflow_error        high                     0.6924   
workflow_error        constraint_error      low                      0.2278   
grounding_state_error workflow_error        medium                   0.4969   
                                            high                     0.7101   
                      constraint_error      low                      0.1354   
                                            high                     0.6701   
workflow_error        grounding_state_error low                      0.2578   
                      tool_use_error        low                      0.1660   
grounding_state_error tool_use_error        low    

In [39]:
# ============================================================
# 36. Prediction pair × semantic confidence
# ============================================================

pair_confidence = (
    conditional_df
    .groupby(
        [
            "semantic_pred_name",
            "transition_pred_name",
            "semantic_confidence_bin",
        ],
        observed=True,
    )
    .agg(
        count=(
            "trust_transition",
            "size",
        ),

        transition_wins=(
            "trust_transition",
            "sum",
        ),

        mean_semantic_confidence=(
            "semantic_confidence",
            "mean",
        ),
    )
)

pair_confidence[
    "transition_win_rate"
] = (
    pair_confidence[
        "transition_wins"
    ]
    /
    pair_confidence[
        "count"
    ]
)

display(
    pair_confidence[
        pair_confidence[
            "count"
        ] >= 5
    ]
    .sort_values(
        "transition_win_rate",
        ascending=False,
    )
    .round(4)
)

count  \
semantic_pred_name    transition_pred_name  semantic_confidence_bin          
tool_use_error        constraint_error      medium                       7   
constraint_error      workflow_error        low                          6   
workflow_error        constraint_error      medium                      10   
                                            high                         8   
                                            low                         10   
tool_use_error        workflow_error        low                          8   
                                            medium                      10   
                                            high                        13   
grounding_state_error workflow_error        low                          8   
                      constraint_error      low                          6   
                                            high                         6   
constraint_error      workflow_error        high                         9   
grounding_state_error workflow_error        high                         5   
workflow_error        grounding_state_error high                         8   
grounding_state_error workflow_error        medium                      10   
                      tool_use_error        low                          5   
                      constraint_error      medium                       5   

                                                                     transition_wins  \
semantic_pred_name    transition_pred_name  semantic_confidence_bin                    
tool_use_error        constraint_error      medium                                 6   
constraint_error      workflow_error        low                                    4   
workflow_error        constraint_error      medium                                 6   
                                            high                                   4   
                                            low                                    5   
tool_use_error        workflow_error        low                                    4   
                                            medium                                 5   
                                            high                                   6   
grounding_state_error workflow_error        low                                    3   
                      constraint_error      low                                    2   
                                            high                                   2   
constraint_error      workflow_error        high                                   3   
grounding_state_error workflow_error        high                                   1   
workflow_error        grounding_state_error high                                   1   
grounding_state_error workflow_error        medium                                 1   
                      tool_use_error        low                                    0   
                      constraint_error      medium                                 0   

                                                                     mean_semantic_confidence  \
semantic_pred_name    transition_pred_name  semantic_confidence_bin                             
tool_use_error        constraint_error      medium                                     0.4585   
constraint_error      workflow_error        low                                        0.3731   
workflow_error        constraint_error      medium                                     0.4702   
                                            high                                       0.5546   
                                            low                                        0.3745   
tool_use_error        workflow_error        low                                        0.3528   
                                            medium                                     0.4726   
                                        

In [40]:
# ============================================================
# 37. History length × similarity
# ============================================================

history_similarity = (
    conditional_df
    .groupby(
        [
            "history_length_bin",
            "similarity_bin",
        ],
        observed=True,
    )
    .agg(
        count=(
            "trust_transition",
            "size",
        ),

        transition_wins=(
            "trust_transition",
            "sum",
        ),
    )
)

history_similarity[
    "transition_win_rate"
] = (
    history_similarity[
        "transition_wins"
    ]
    /
    history_similarity[
        "count"
    ]
)

display(
    history_similarity
    .sort_values(
        "transition_win_rate",
        ascending=False,
    )
    .round(4)
)

count  transition_wins  transition_win_rate
history_length_bin similarity_bin                                             
long_11_plus       medium             24               15               0.6250
short_0_3          medium             10                5               0.5000
                   high                2                1               0.5000
medium_4_10        medium             27               13               0.4815
                   high               36               15               0.4167
                   low                25                7               0.2800
short_0_3          low                28                7               0.2500
long_11_plus       high               23                5               0.2174
                   low                 8                1               0.1250

In [41]:
# ============================================================
# 38. Standardized effect sizes
# ============================================================

numeric_utility_features = [
    "current_tminus1_cosine",
    "current_tminus2_cosine",
    "current_tminus3_cosine",
    "recent_similarity_mean",
    "recent_similarity_max",
    "history_transition_1_cosine",
    "history_transition_2_cosine",

    "semantic_confidence",
    "semantic_margin",
    "semantic_entropy",

    "transition_confidence",
    "transition_margin",
    "transition_entropy",

    "confidence_delta",
    "history_event_count",
]


def cohens_d(
    positive,
    negative,
):
    positive = np.asarray(
        positive,
        dtype=float,
    )

    negative = np.asarray(
        negative,
        dtype=float,
    )

    n1 = len(positive)
    n0 = len(negative)

    if n1 < 2 or n0 < 2:
        return np.nan

    var1 = positive.var(
        ddof=1
    )

    var0 = negative.var(
        ddof=1
    )

    pooled = np.sqrt(
        (
            (n1 - 1) * var1
            +
            (n0 - 1) * var0
        )
        /
        (
            n1 + n0 - 2
        )
    )

    if pooled == 0:
        return 0.0

    return (
        positive.mean()
        - negative.mean()
    ) / pooled


effect_rows = []

for feature in numeric_utility_features:

    transition_win_values = (
        conditional_df.loc[
            conditional_df[
                "trust_transition"
            ] == 1,
            feature,
        ]
    )

    semantic_win_values = (
        conditional_df.loc[
            conditional_df[
                "trust_transition"
            ] == 0,
            feature,
        ]
    )

    effect_rows.append({
        "feature":
            feature,

        "transition_win_mean":
            transition_win_values.mean(),

        "semantic_win_mean":
            semantic_win_values.mean(),

        "difference":
            (
                transition_win_values.mean()
                - semantic_win_values.mean()
            ),

        "cohens_d":
            cohens_d(
                transition_win_values,
                semantic_win_values,
            ),
    })


effect_size_df = pd.DataFrame(
    effect_rows
)

display(
    effect_size_df
    .assign(
        absolute_d=lambda x:
            x[
                "cohens_d"
            ].abs()
    )
    .sort_values(
        "absolute_d",
        ascending=False,
    )
    .drop(
        columns="absolute_d"
    )
    .round(4)
)

,feature,transition_win_mean,semantic_win_mean,difference,cohens_d
4,recent_similarity_max,0.6530,0.5767,0.0763,0.2943
1,current_tminus2_cosine,0.5254,0.4420,0.0834,0.2835
9,semantic_entropy,1.0899,1.1333,-0.0434,-0.2634
14,history_event_count,8.3188,11.0789,-2.7601,-0.2502
12,transition_entropy,1.1781,1.2165,-0.0384,-0.2490
3,recent_similarity_mean,0.4770,0.4272,0.0498,0.2172
0,current_tminus1_cosine,0.5204,0.4687,0.0516,0.1966
8,semantic_margin,0.1144,0.1366,-0.0222,-0.1892
10,transition_confidence,0.4411,0.4272,0.0138,0.1732
13,confidence_delta,-0.0267,-0.0379,0.0111,0.1213


In [42]:
# ============================================================
# 39. Shallow decision tree for utility rules
# ============================================================

from sklearn.tree import (
    DecisionTreeClassifier,
    export_text,
)

tree_features = [
    "recent_similarity_mean",
    "current_tminus1_cosine",
    "current_tminus2_cosine",

    "semantic_confidence",
    "semantic_margin",

    "transition_confidence",
    "transition_margin",

    "history_event_count",
]


tree_X = (
    conditional_df[
        tree_features
    ]
    .to_numpy()
)

tree_y = (
    conditional_df[
        "trust_transition"
    ]
    .to_numpy()
)


utility_tree = DecisionTreeClassifier(
    max_depth=3,
    min_samples_leaf=10,
    class_weight="balanced",
    random_state=42,
)

utility_tree.fit(
    tree_X,
    tree_y,
)


print(
    export_text(
        utility_tree,
        feature_names=tree_features,
        decimals=3,
    )
)

|--- history_event_count <= 23.500
|   |--- recent_similarity_mean <= 0.352
|   |   |--- transition_confidence <= 0.415
|   |   |   |--- class: 0
|   |   |--- transition_confidence >  0.415
|   |   |   |--- class: 1
|   |--- recent_similarity_mean >  0.352
|   |   |--- transition_margin <= 0.257
|   |   |   |--- class: 1
|   |   |--- transition_margin >  0.257
|   |   |   |--- class: 0
|--- history_event_count >  23.500
|   |--- class: 0



In [43]:
# ============================================================
# 40. Explicit interaction features
# ============================================================

interaction_df = conditional_df.copy()

interaction_df["similarity_x_sem_conf"] = (
    interaction_df["recent_similarity_mean"]
    * interaction_df["semantic_confidence"]
)

interaction_df["similarity_x_sem_margin"] = (
    interaction_df["recent_similarity_mean"]
    * interaction_df["semantic_margin"]
)

interaction_df["similarity_x_trans_conf"] = (
    interaction_df["recent_similarity_mean"]
    * interaction_df["transition_confidence"]
)

interaction_df["similarity_x_trans_margin"] = (
    interaction_df["recent_similarity_mean"]
    * interaction_df["transition_margin"]
)

# log transform because history length is strongly skewed
interaction_df["log_history_count"] = np.log1p(
    interaction_df["history_event_count"]
)

interaction_df["similarity_x_log_history"] = (
    interaction_df["recent_similarity_mean"]
    * interaction_df["log_history_count"]
)

# Capture the observed non-monotonicity.
interaction_df["similarity_squared"] = (
    interaction_df["recent_similarity_mean"] ** 2
)

interaction_df["prediction_pair"] = (
    interaction_df["semantic_pred_name"].astype(str)
    + "__TO__"
    + interaction_df["transition_pred_name"].astype(str)
)

In [44]:
# ============================================================
# 41. Interaction-aware router feature set
# ============================================================

interaction_numeric_features = [
    "recent_similarity_mean",
    "recent_similarity_max",
    "similarity_squared",

    "current_tminus1_cosine",
    "current_tminus2_cosine",

    "semantic_confidence",
    "semantic_margin",
    "semantic_entropy",

    "transition_confidence",
    "transition_margin",
    "transition_entropy",

    "log_history_count",

    "similarity_x_sem_conf",
    "similarity_x_sem_margin",
    "similarity_x_trans_conf",
    "similarity_x_trans_margin",
    "similarity_x_log_history",
]

interaction_categorical_features = [
    "semantic_pred_name",
    "transition_pred_name",
    "prediction_pair",
]

X_interaction = interaction_df[
    interaction_numeric_features
    + interaction_categorical_features
].reset_index(drop=True)

y_interaction = (
    interaction_df["trust_transition"]
    .to_numpy()
    .astype(int)
)

groups_interaction = np.asarray(
    router_groups_all
)

print("X:", X_interaction.shape)
print("y:", y_interaction.shape)

print(
    "Transition wins:",
    y_interaction.sum()
)

print(
    "Semantic wins:",
    (y_interaction == 0).sum()
)

X: (183, 20)
y: (183,)
Transition wins: 69
Semantic wins: 114


In [45]:
# ============================================================
# 42. Interaction preprocessing
# ============================================================

interaction_preprocessor = ColumnTransformer(
    transformers=[
        (
            "numeric",
            StandardScaler(),
            interaction_numeric_features,
        ),
        (
            "categorical",
            OneHotEncoder(
                handle_unknown="ignore"
            ),
            interaction_categorical_features,
        ),
    ]
)

In [46]:
# ============================================================
# 43. Group-safe interaction-router CV
# ============================================================

interaction_C_values = [
    0.01,
    0.03,
    0.1,
    0.3,
    1.0,
    3.0,
]

interaction_rows = []

interaction_cv = StratifiedGroupKFold(
    n_splits=5,
    shuffle=True,
    random_state=42,
)

for C in interaction_C_values:

    fold_rows = []

    for tr_idx, va_idx in interaction_cv.split(
        X_interaction,
        y_interaction,
        groups=groups_interaction,
    ):

        model = Pipeline([
            (
                "preprocessor",
                interaction_preprocessor,
            ),
            (
                "classifier",
                LogisticRegression(
                    C=C,
                    max_iter=5000,
                    class_weight=None,
                    random_state=42,
                ),
            ),
        ])

        model.fit(
            X_interaction.iloc[tr_idx],
            y_interaction[tr_idx],
        )

        pred = model.predict(
            X_interaction.iloc[va_idx]
        )

        fold_rows.append({
            "accuracy": accuracy_score(
                y_interaction[va_idx],
                pred,
            ),

            "balanced_accuracy":
                balanced_accuracy_score(
                    y_interaction[va_idx],
                    pred,
                ),

            "macro_f1": f1_score(
                y_interaction[va_idx],
                pred,
                average="macro",
                zero_division=0,
            ),
        })

    fold_df = pd.DataFrame(fold_rows)

    interaction_rows.append({
        "C": C,

        "accuracy":
            fold_df["accuracy"].mean(),

        "balanced_accuracy":
            fold_df[
                "balanced_accuracy"
            ].mean(),

        "macro_f1":
            fold_df["macro_f1"].mean(),

        "macro_f1_std":
            fold_df["macro_f1"].std(),
    })

interaction_cv_results = pd.DataFrame(
    interaction_rows
)

display(
    interaction_cv_results
    .sort_values(
        "macro_f1",
        ascending=False,
    )
    .round(4)
)

,C,accuracy,balanced_accuracy,macro_f1,macro_f1_std
2,0.10,0.6283,0.5529,0.5398,0.1424
3,0.30,0.6059,0.5459,0.5388,0.0883
4,1.00,0.5786,0.5260,0.5175,0.0946
5,3.00,0.5621,0.5097,0.5011,0.1106
1,0.03,0.5958,0.4933,0.4253,0.0977
0,0.01,0.6066,0.4870,0.3773,0.0155


In [48]:
# ============================================================
# Define failure-family class names
# ============================================================

class_names = [
    "workflow_error",
    "constraint_error",
    "tool_use_error",
    "grounding_state_error",
    "reasoning_value_error",
]

print("Classes:")
for i, name in enumerate(class_names):
    print(i, name)

Classes:
0 workflow_error
1 constraint_error
2 tool_use_error
3 grounding_state_error
4 reasoning_value_error


In [52]:
# ============================================================
# Find existing OOF prediction variables safely
# ============================================================

global_items = list(globals().items())

for name, obj in global_items:

    if (
        "oof" in name.lower()
        or "semantic" in name.lower()
        or "transition" in name.lower()
        or "pred" in name.lower()
    ):
        try:
            arr = np.asarray(obj)

            if (
                arr.ndim >= 1
                and arr.shape[0] == len(y_train)
            ):
                print(
                    f"{name:45s}",
                    arr.shape,
                )

        except Exception:
            pass

oof_sem_prob                                  (1489, 5)
oof_trans_prob                                (1489, 5)
oof_sem_pred                                  (1489,)
oof_trans_pred                                (1489,)
compact_router_oof_choice                     (1489,)
compact_routed_pred                           (1489,)
semantic_correct_final                        (1489,)


In [53]:
oof_semantic_pred = oof_sem_pred
oof_transition_pred = oof_trans_pred

In [54]:
# ============================================================
# NEXT RESEARCH:
# Class-conditional trajectory utility
# ============================================================

family_utility = []

for family in class_names:

    family_id = class_names.index(family)

    true_family = (
        y_train == family_id
    )

    sem_family = (
        oof_semantic_pred == family_id
    )

    trans_family = (
        oof_transition_pred == family_id
    )

    # For examples actually belonging to this family
    mask = true_family

    sem_recall = sem_family[mask].mean()
    trans_recall = trans_family[mask].mean()

    # trajectory rescues this family
    rescues = (
        (~sem_family[mask])
        & trans_family[mask]
    ).sum()

    # trajectory destroys a correct semantic prediction
    breaks = (
        sem_family[mask]
        & (~trans_family[mask])
    ).sum()

    family_utility.append({
        "failure_family": family,
        "support": int(mask.sum()),
        "semantic_recall": sem_recall,
        "transition_recall": trans_recall,
        "recall_delta": (
            trans_recall - sem_recall
        ),
        "rescues": int(rescues),
        "breaks": int(breaks),
        "net_rescues": int(rescues - breaks),
    })

family_utility_df = pd.DataFrame(
    family_utility
).sort_values(
    "recall_delta",
    ascending=False,
)

display(
    family_utility_df.round(4)
)

,failure_family,support,semantic_recall,transition_recall,recall_delta,rescues,breaks,net_rescues
1,constraint_error,317,0.3817,0.4416,0.0599,26,7,19
0,workflow_error,660,0.6455,0.6576,0.0121,28,20,8
3,grounding_state_error,244,0.4590,0.4467,-0.0123,12,15,-3
2,tool_use_error,237,0.4515,0.4219,-0.0295,3,10,-7
4,reasoning_value_error,31,0.5161,0.4516,-0.0645,0,2,-2


In [56]:
# ============================================================
# 49. Per-family semantic vs transition confusion shifts
# ============================================================

from sklearn.metrics import (
    confusion_matrix,
)

cm_sem = confusion_matrix(
    y_train,
    oof_semantic_pred,
    labels=np.arange(
        len(class_names)
    ),
)

cm_trans = confusion_matrix(
    y_train,
    oof_transition_pred,
    labels=np.arange(
        len(class_names)
    ),
)


for i, family in enumerate(
    class_names
):

    comparison = pd.DataFrame({
        "predicted_family":
            class_names,

        "semantic_count":
            cm_sem[i],

        "transition_count":
            cm_trans[i],
    })

    comparison[
        "delta_transition_minus_semantic"
    ] = (
        comparison[
            "transition_count"
        ]
        -
        comparison[
            "semantic_count"
        ]
    )

    print(
        "\n"
        + "=" * 80
    )

    print(
        "TRUE FAMILY:",
        family
    )

    print(
        "=" * 80
    )

    display(
        comparison
        .sort_values(
            "delta_transition_minus_semantic",
            ascending=False,
        )
    )


TRUE FAMILY: workflow_error


,predicted_family,semantic_count,transition_count,delta_transition_minus_semantic
0,workflow_error,426,434,8
3,grounding_state_error,53,57,4
4,reasoning_value_error,2,4,2
1,constraint_error,102,99,-3
2,tool_use_error,77,66,-11



TRUE FAMILY: constraint_error


,predicted_family,semantic_count,transition_count,delta_transition_minus_semantic
1,constraint_error,121,140,19
0,workflow_error,114,116,2
4,reasoning_value_error,3,1,-2
2,tool_use_error,25,20,-5
3,grounding_state_error,54,40,-14



TRUE FAMILY: tool_use_error


,predicted_family,semantic_count,transition_count,delta_transition_minus_semantic
0,workflow_error,84,91,7
1,constraint_error,23,28,5
4,reasoning_value_error,2,2,0
3,grounding_state_error,21,16,-5
2,tool_use_error,107,100,-7



TRUE FAMILY: grounding_state_error


,predicted_family,semantic_count,transition_count,delta_transition_minus_semantic
0,workflow_error,65,68,3
1,constraint_error,49,52,3
4,reasoning_value_error,5,4,-1
2,tool_use_error,13,11,-2
3,grounding_state_error,112,109,-3



TRUE FAMILY: reasoning_value_error


,predicted_family,semantic_count,transition_count,delta_transition_minus_semantic
0,workflow_error,3,5,2
1,constraint_error,5,6,1
2,tool_use_error,2,2,0
3,grounding_state_error,5,4,-1
4,reasoning_value_error,16,14,-2


In [57]:
# ============================================================
# 50. Which classes actually benefit from trajectory?
# ============================================================

family_utility_df[
    "trajectory_effect"
] = np.select(
    [
        family_utility_df[
            "net_rescues"
        ] > 0,

        family_utility_df[
            "net_rescues"
        ] < 0,
    ],
    [
        "beneficial",
        "harmful",
    ],
    default="neutral",
)


display(
    family_utility_df[
        [
            "failure_family",
            "support",
            "semantic_recall",
            "transition_recall",
            "recall_delta",
            "rescues",
            "breaks",
            "net_rescues",
            "trajectory_effect",
        ]
    ]
    .round(4)
)

,failure_family,support,semantic_recall,transition_recall,recall_delta,rescues,breaks,net_rescues,trajectory_effect
1,constraint_error,317,0.3817,0.4416,0.0599,26,7,19,beneficial
0,workflow_error,660,0.6455,0.6576,0.0121,28,20,8,beneficial
3,grounding_state_error,244,0.4590,0.4467,-0.0123,12,15,-3,harmful
2,tool_use_error,237,0.4515,0.4219,-0.0295,3,10,-7,harmful
4,reasoning_value_error,31,0.5161,0.4516,-0.0645,0,2,-2,harmful


In [58]:
# ============================================================
# 51. Targeted constraint-error correction dataset
# ============================================================

constraint_id = class_names.index("constraint_error")

# Candidate cases:
# semantic model did NOT already predict constraint_error
candidate_mask = (
    oof_sem_pred != constraint_id
)

candidate_idx = np.where(candidate_mask)[0]

# Binary target:
# should we override semantic prediction -> constraint_error?
y_constraint_override = (
    y_train[candidate_mask] == constraint_id
).astype(int)

print("Candidate examples:", len(candidate_idx))
print(
    "True constraint overrides:",
    y_constraint_override.sum()
)
print(
    "Positive prevalence:",
    y_constraint_override.mean()
)

# What does the trajectory expert do on these?
traj_suggests_constraint = (
    oof_trans_pred[candidate_mask]
    == constraint_id
)

print(
    "\nTrajectory suggests constraint:",
    traj_suggests_constraint.sum()
)

print(
    "Correct trajectory constraint suggestions:",
    (
        traj_suggests_constraint
        & (y_constraint_override == 1)
    ).sum()
)

print(
    "Incorrect trajectory constraint suggestions:",
    (
        traj_suggests_constraint
        & (y_constraint_override == 0)
    ).sum()
)

Candidate examples: 1189
True constraint overrides: 196
Positive prevalence: 0.1648444070647603

Trajectory suggests constraint: 56
Correct trajectory constraint suggestions: 26
Incorrect trajectory constraint suggestions: 30


In [59]:
# ============================================================
# 52. Baseline targeted correction rule
# ============================================================

from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score,
)

rule_pred = (
    oof_trans_pred[candidate_mask]
    == constraint_id
).astype(int)

print(
    "Override precision:",
    precision_score(
        y_constraint_override,
        rule_pred,
        zero_division=0,
    )
)

print(
    "Override recall:",
    recall_score(
        y_constraint_override,
        rule_pred,
        zero_division=0,
    )
)

print(
    "Override F1:",
    f1_score(
        y_constraint_override,
        rule_pred,
        zero_division=0,
    )
)

Override precision: 0.4642857142857143
Override recall: 0.1326530612244898
Override F1: 0.20634920634920634


In [60]:
# ============================================================
# 53. Apply targeted constraint correction
# ============================================================

constraint_corrected_pred = (
    oof_sem_pred.copy()
)

override_mask = (
    (oof_sem_pred != constraint_id)
    &
    (oof_trans_pred == constraint_id)
)

constraint_corrected_pred[
    override_mask
] = constraint_id

print(
    "Overrides:",
    override_mask.sum()
)

print(
    "Semantic accuracy:",
    (
        oof_sem_pred == y_train
    ).mean()
)

print(
    "Corrected accuracy:",
    (
        constraint_corrected_pred
        == y_train
    ).mean()
)

print(
    "Accuracy delta:",
    (
        constraint_corrected_pred
        == y_train
    ).mean()
    -
    (
        oof_sem_pred
        == y_train
    ).mean()
)

Overrides: 56
Semantic accuracy: 0.5251846877098724
Corrected accuracy: 0.5325721961047682
Accuracy delta: 0.007387508394895881


In [61]:
# ============================================================
# 54. Full targeted-correction evaluation
# ============================================================

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    classification_report,
)

def evaluate_preds(y_true, pred):
    return {
        "accuracy":
            accuracy_score(y_true, pred),

        "balanced_accuracy":
            balanced_accuracy_score(
                y_true,
                pred
            ),

        "macro_f1":
            f1_score(
                y_true,
                pred,
                average="macro",
            ),

        "weighted_f1":
            f1_score(
                y_true,
                pred,
                average="weighted",
            ),
    }


comparison = pd.DataFrame([
    {
        "model": "semantic",
        **evaluate_preds(
            y_train,
            oof_sem_pred,
        ),
    },
    {
        "model":
            "constraint_targeted_correction",
        **evaluate_preds(
            y_train,
            constraint_corrected_pred,
        ),
    },
    {
        "model": "transition",
        **evaluate_preds(
            y_train,
            oof_trans_pred,
        ),
    },
    {
        "model": "compact_router",
        **evaluate_preds(
            y_train,
            compact_routed_pred,
        ),
    },
])

display(
    comparison.round(4)
)

print(
    classification_report(
        y_train,
        constraint_corrected_pred,
        target_names=class_names,
        digits=4,
    )
)

,model,accuracy,balanced_accuracy,macro_f1,weighted_f1
0,semantic,0.5252,0.4908,0.4974,0.5231
1,constraint_targeted_correction,0.5326,0.4985,0.5056,0.5327
2,transition,0.5353,0.4839,0.4981,0.5323
3,compact_router,0.5353,0.4918,0.5008,0.5335


                       precision    recall  f1-score   support

       workflow_error     0.6310    0.6348    0.6329       660
     constraint_error     0.4129    0.4637    0.4368       317
       tool_use_error     0.4977    0.4473    0.4711       237
grounding_state_error     0.4605    0.4303    0.4449       244
reasoning_value_error     0.5714    0.5161    0.5424        31

             accuracy                         0.5326      1489
            macro avg     0.5147    0.4985    0.5056      1489
         weighted avg     0.5342    0.5326    0.5327      1489



In [62]:
# ============================================================
# 55. Analyze trajectory-proposed constraint overrides
# ============================================================

constraint_id = class_names.index(
    "constraint_error"
)

proposal_mask = (
    (oof_sem_pred != constraint_id)
    &
    (oof_trans_pred == constraint_id)
)

proposal_idx = np.where(
    proposal_mask
)[0]

# 1 = trajectory correction is correct
# 0 = semantic/default should be retained
proposal_target = (
    y_train[proposal_mask]
    == constraint_id
).astype(int)

print(
    "Constraint proposals:",
    len(proposal_idx)
)

print(
    "Good overrides:",
    proposal_target.sum()
)

print(
    "Bad overrides:",
    len(proposal_target)
    - proposal_target.sum()
)

print(
    "Good override prevalence:",
    proposal_target.mean()
)

Constraint proposals: 56
Good overrides: 26
Bad overrides: 30
Good override prevalence: 0.4642857142857143


In [63]:
# ============================================================
# 56. Probability characteristics of good vs bad overrides
# ============================================================

eps = 1e-12

sem_prob_prop = (
    oof_sem_prob[proposal_mask]
)

trans_prob_prop = (
    oof_trans_prob[proposal_mask]
)

semantic_confidence = (
    sem_prob_prop.max(axis=1)
)

transition_confidence = (
    trans_prob_prop.max(axis=1)
)

semantic_constraint_prob = (
    sem_prob_prop[:, constraint_id]
)

transition_constraint_prob = (
    trans_prob_prop[:, constraint_id]
)

constraint_prob_gain = (
    transition_constraint_prob
    - semantic_constraint_prob
)


def probability_margin(p):
    sorted_p = np.sort(
        p,
        axis=1,
    )

    return (
        sorted_p[:, -1]
        - sorted_p[:, -2]
    )


semantic_margin = probability_margin(
    sem_prob_prop
)

transition_margin = probability_margin(
    trans_prob_prop
)


semantic_entropy = -np.sum(
    sem_prob_prop
    * np.log(
        sem_prob_prop + eps
    ),
    axis=1,
)

transition_entropy = -np.sum(
    trans_prob_prop
    * np.log(
        trans_prob_prop + eps
    ),
    axis=1,
)


proposal_analysis = pd.DataFrame({
    "target":
        proposal_target,

    "semantic_pred":
        oof_sem_pred[
            proposal_mask
        ],

    "semantic_confidence":
        semantic_confidence,

    "semantic_constraint_prob":
        semantic_constraint_prob,

    "semantic_margin":
        semantic_margin,

    "semantic_entropy":
        semantic_entropy,

    "transition_confidence":
        transition_confidence,

    "transition_constraint_prob":
        transition_constraint_prob,

    "transition_margin":
        transition_margin,

    "transition_entropy":
        transition_entropy,

    "constraint_prob_gain":
        constraint_prob_gain,
})


display(
    proposal_analysis
    .groupby("target")
    .agg([
        "count",
        "mean",
        "median",
        "std",
    ])
    .round(4)
)

semantic_pred                        semantic_confidence          \
               count    mean median     std               count    mean   
target                                                                    
0                 30  1.5667    2.0  1.4308                  30  0.4537   
1                 26  1.0000    0.0  1.2329                  26  0.4652   

                       semantic_constraint_prob          ...  \
        median     std                    count    mean  ...   
target                                                   ...   
0       0.4565  0.0918                       30  0.3025  ...   
1       0.4668  0.0833                       26  0.3432  ...   

       transition_margin         transition_entropy                          \
                  median     std              count    mean  median     std   
target                                                                        
0                 0.0462  0.0526                 30  1.2698  1.2736  0.0897   
1                 0.0911  0.0684                 26  1.2049  1.1965  0.1232   

       constraint_prob_gain                          
                      count    mean  median     std  
target                                               
0                        30  0.0805  0.0758  0.0690  
1                        26  0.0880  0.0830  0.0521  

[2 rows x 40 columns]

In [64]:
# ============================================================
# 57. Constraint correction utility by semantic prediction
# ============================================================

proposal_analysis[
    "semantic_pred_name"
] = [
    class_names[i]
    for i in proposal_analysis[
        "semantic_pred"
    ]
]


proposal_by_semantic = (
    proposal_analysis
    .groupby(
        "semantic_pred_name"
    )
    .agg(
        count=(
            "target",
            "size",
        ),
        good_overrides=(
            "target",
            "sum",
        ),
        override_precision=(
            "target",
            "mean",
        ),
        mean_transition_constraint_prob=(
            "transition_constraint_prob",
            "mean",
        ),
        mean_constraint_prob_gain=(
            "constraint_prob_gain",
            "mean",
        ),
        mean_semantic_confidence=(
            "semantic_confidence",
            "mean",
        ),
    )
    .sort_values(
        "override_precision",
        ascending=False,
    )
)

display(
    proposal_by_semantic
    .round(4)
)

,count,good_overrides,override_precision,mean_transition_constraint_prob,mean_constraint_prob_gain,mean_semantic_confidence
semantic_pred_name,,,,,,
tool_use_error,11,7,0.6364,0.3940,0.0970,0.4298
workflow_error,28,15,0.5357,0.4239,0.0781,0.4602
grounding_state_error,17,4,0.2353,0.3822,0.0853,0.4760


In [65]:
# ============================================================
# 58. Threshold trajectory constraint probability
# ============================================================

threshold_rows = []

for threshold in np.arange(
    0.30,
    0.91,
    0.05,
):

    accept_local = (
        transition_constraint_prob
        >= threshold
    )

    final_pred = (
        oof_sem_pred.copy()
    )

    accepted_idx = (
        proposal_idx[
            accept_local
        ]
    )

    final_pred[
        accepted_idx
    ] = constraint_id

    accepted_targets = (
        proposal_target[
            accept_local
        ]
    )

    n_accept = int(
        accept_local.sum()
    )

    good = int(
        accepted_targets.sum()
    )

    bad = (
        n_accept - good
    )

    metrics = evaluate_preds(
        y_train,
        final_pred,
    )

    threshold_rows.append({
        "threshold":
            threshold,

        "overrides":
            n_accept,

        "good":
            good,

        "bad":
            bad,

        "net":
            good - bad,

        "precision":
            (
                good / n_accept
                if n_accept
                else np.nan
            ),

        **metrics,
    })


constraint_threshold_df = (
    pd.DataFrame(
        threshold_rows
    )
    .sort_values(
        [
            "macro_f1",
            "balanced_accuracy",
        ],
        ascending=False,
    )
)

display(
    constraint_threshold_df
    .round(4)
)

,threshold,overrides,good,bad,net,precision,accuracy,balanced_accuracy,macro_f1,weighted_f1
2,0.40,28,19,9,10,0.6786,0.5346,0.5002,0.5070,0.5339
1,0.35,43,22,21,1,0.5116,0.5332,0.4995,0.5063,0.5332
0,0.30,56,26,30,-4,0.4643,0.5326,0.4985,0.5056,0.5327
3,0.45,14,10,4,6,0.7143,0.5299,0.4956,0.5023,0.5287
4,0.50,4,3,1,2,0.7500,0.5265,0.4923,0.4990,0.5247
5,0.55,0,0,0,0,NaN,0.5252,0.4908,0.4974,0.5231
6,0.60,0,0,0,0,NaN,0.5252,0.4908,0.4974,0.5231
7,0.65,0,0,0,0,NaN,0.5252,0.4908,0.4974,0.5231
8,0.70,0,0,0,0,NaN,0.5252,0.4908,0.4974,0.5231
9,0.75,0,0,0,0,NaN,0.5252,0.4908,0.4974,0.5231


In [66]:
# ============================================================
# 59. Conditional constraint-correction rules
# ============================================================

rows = []

source_families = [
    "workflow_error",
    "tool_use_error",
    "grounding_state_error",
]

thresholds = np.arange(
    0.30,
    0.51,
    0.025,
)

# proposal_analysis corresponds exactly to proposal_idx
source_names = np.array([
    class_names[i]
    for i in oof_sem_pred[proposal_mask]
])


def evaluate_constraint_rule(
    threshold,
    allowed_sources,
):
    # Local mask over the 56 trajectory constraint proposals
    accept_local = (
        (transition_constraint_prob >= threshold)
        &
        np.isin(
            source_names,
            allowed_sources,
        )
    )

    final_pred = oof_sem_pred.copy()

    accepted_idx = proposal_idx[
        accept_local
    ]

    final_pred[
        accepted_idx
    ] = constraint_id

    accepted_target = proposal_target[
        accept_local
    ]

    good = int(
        accepted_target.sum()
    )

    total = int(
        accept_local.sum()
    )

    bad = total - good

    metrics = evaluate_preds(
        y_train,
        final_pred,
    )

    return {
        "threshold": threshold,
        "allowed_sources":
            "+".join(allowed_sources),
        "overrides": total,
        "good": good,
        "bad": bad,
        "net": good - bad,
        "precision":
            good / total
            if total > 0
            else np.nan,
        **metrics,
    }


# ------------------------------------------------------------
# Candidate source-family policies
# ------------------------------------------------------------

source_policies = [
    ["workflow_error"],
    ["tool_use_error"],
    ["grounding_state_error"],

    [
        "workflow_error",
        "tool_use_error",
    ],

    [
        "workflow_error",
        "grounding_state_error",
    ],

    [
        "tool_use_error",
        "grounding_state_error",
    ],

    [
        "workflow_error",
        "tool_use_error",
        "grounding_state_error",
    ],
]


for policy in source_policies:
    for threshold in thresholds:

        rows.append(
            evaluate_constraint_rule(
                threshold,
                policy,
            )
        )


conditional_constraint_df = (
    pd.DataFrame(rows)
    .sort_values(
        [
            "macro_f1",
            "balanced_accuracy",
            "accuracy",
        ],
        ascending=False,
    )
    .reset_index(drop=True)
)

display(
    conditional_constraint_df
    .head(20)
    .round(4)
)

,threshold,allowed_sources,overrides,good,bad,net,precision,accuracy,balanced_accuracy,macro_f1,weighted_f1
0,0.300,workflow_error+tool_use_error,39,22,17,5,0.5641,0.5346,0.5017,0.5081,0.5347
1,0.325,workflow_error+tool_use_error,36,21,15,6,0.5833,0.5346,0.5013,0.5077,0.5346
2,0.375,workflow_error+tool_use_error,27,19,8,11,0.7037,0.5346,0.5007,0.5072,0.5341
3,0.400,workflow_error+tool_use_error,22,17,5,12,0.7727,0.5346,0.5006,0.5071,0.5339
4,0.400,workflow_error+tool_use_error+grounding_state_...,28,19,9,10,0.6786,0.5346,0.5002,0.5070,0.5339
5,0.350,workflow_error+tool_use_error,32,19,13,6,0.5938,0.5332,0.5001,0.5064,0.5331
6,0.350,workflow_error+tool_use_error+grounding_state_...,43,22,21,1,0.5116,0.5332,0.4995,0.5063,0.5332
7,0.375,workflow_error+tool_use_error+grounding_state_...,34,21,13,8,0.6176,0.5339,0.4995,0.5063,0.5335
8,0.325,workflow_error+tool_use_error+grounding_state_...,52,25,27,-2,0.4808,0.5332,0.4989,0.5060,0.5333
9,0.300,workflow_error+tool_use_error+grounding_state_...,56,26,30,-4,0.4643,0.5326,0.4985,0.5056,0.5327


In [67]:
# ============================================================
# 60. Best conditional correction vs previous models
# ============================================================

best_rule = (
    conditional_constraint_df.iloc[0]
)

print("BEST RULE")
print(best_rule)

best_sources = (
    best_rule[
        "allowed_sources"
    ].split("+")
)

best_threshold = float(
    best_rule["threshold"]
)

best_accept_local = (
    (
        transition_constraint_prob
        >= best_threshold
    )
    &
    np.isin(
        source_names,
        best_sources,
    )
)

best_constraint_pred = (
    oof_sem_pred.copy()
)

best_constraint_pred[
    proposal_idx[
        best_accept_local
    ]
] = constraint_id


final_comparison = pd.DataFrame([
    {
        "model":
            "semantic",
        **evaluate_preds(
            y_train,
            oof_sem_pred,
        ),
    },

    {
        "model":
            "transition",
        **evaluate_preds(
            y_train,
            oof_trans_pred,
        ),
    },

    {
        "model":
            "compact_router",
        **evaluate_preds(
            y_train,
            compact_routed_pred,
        ),
    },

    {
        "model":
            "constraint_threshold_0.40",
        **evaluate_preds(
            y_train,
            # reconstruct 0.40 version
            np.where(
                (
                    (oof_sem_pred != constraint_id)
                    &
                    (oof_trans_pred == constraint_id)
                    &
                    (
                        oof_trans_prob[
                            :, constraint_id
                        ] >= 0.40
                    )
                ),
                constraint_id,
                oof_sem_pred,
            ),
        ),
    },

    {
        "model":
            "conditional_constraint_rule",
        **evaluate_preds(
            y_train,
            best_constraint_pred,
        ),
    },
])

display(
    final_comparison
    .sort_values(
        "macro_f1",
        ascending=False,
    )
    .round(4)
)

print(
    "\nBest rule overrides:",
    best_accept_local.sum(),
)

print(
    "Good:",
    proposal_target[
        best_accept_local
    ].sum(),
)

print(
    "Bad:",
    (
        best_accept_local.sum()
        -
        proposal_target[
            best_accept_local
        ].sum()
    ),
)

BEST RULE
threshold                                      0.3
allowed_sources      workflow_error+tool_use_error
overrides                                       39
good                                            22
bad                                             17
net                                              5
precision                                 0.564103
accuracy                                  0.534587
balanced_accuracy                         0.501671
macro_f1                                  0.508094
weighted_f1                               0.534705
Name: 0, dtype: object


,model,accuracy,balanced_accuracy,macro_f1,weighted_f1
4,conditional_constraint_rule,0.5346,0.5017,0.5081,0.5347
3,constraint_threshold_0.40,0.5346,0.5002,0.5070,0.5339
2,compact_router,0.5353,0.4918,0.5008,0.5335
1,transition,0.5353,0.4839,0.4981,0.5323
0,semantic,0.5252,0.4908,0.4974,0.5231



Best rule overrides: 39
Good: 22
Bad: 17


In [68]:
# ============================================================
# 61. Cross-fitted validation of targeted trajectory correction
# ============================================================

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
)
import numpy as np
import pandas as pd


# ------------------------------------------------------------
# Candidate policies
# ------------------------------------------------------------

thresholds = np.arange(
    0.30,
    0.501,
    0.025,
)

source_policies = [
    ["workflow_error"],
    ["tool_use_error"],
    ["grounding_state_error"],

    ["workflow_error", "tool_use_error"],
    ["workflow_error", "grounding_state_error"],
    ["tool_use_error", "grounding_state_error"],

    [
        "workflow_error",
        "tool_use_error",
        "grounding_state_error",
    ],
]


# ------------------------------------------------------------
# Basic quantities
# ------------------------------------------------------------

semantic_source_names = np.array([
    class_names[i]
    for i in oof_sem_pred
])

transition_constraint_prob_all = (
    oof_trans_prob[:, constraint_id]
)

proposal_all = (
    (oof_sem_pred != constraint_id)
    &
    (oof_trans_pred == constraint_id)
)


# ------------------------------------------------------------
# Apply one correction policy
# ------------------------------------------------------------

def apply_policy(
    indices,
    threshold,
    allowed_sources,
):
    pred = oof_sem_pred[indices].copy()

    accept = (
        proposal_all[indices]
        &
        (
            transition_constraint_prob_all[indices]
            >= threshold
        )
        &
        np.isin(
            semantic_source_names[indices],
            allowed_sources,
        )
    )

    pred[accept] = constraint_id

    return pred, accept


# ------------------------------------------------------------
# Score one policy
# ------------------------------------------------------------

def policy_score(
    indices,
    threshold,
    allowed_sources,
):
    pred, accept = apply_policy(
        indices,
        threshold,
        allowed_sources,
    )

    y = y_train[indices]

    return {
        "accuracy":
            accuracy_score(y, pred),

        "balanced_accuracy":
            balanced_accuracy_score(y, pred),

        "macro_f1":
            f1_score(
                y,
                pred,
                average="macro",
            ),

        "weighted_f1":
            f1_score(
                y,
                pred,
                average="weighted",
            ),

        "overrides":
            int(accept.sum()),
    }


# ------------------------------------------------------------
# Outer cross-fitting
# ------------------------------------------------------------

cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42,
)

crossfit_pred = np.empty_like(
    oof_sem_pred
)

selected_rules = []


for fold, (fit_idx, eval_idx) in enumerate(
    cv.split(
        np.zeros(len(y_train)),
        y_train,
    ),
    start=1,
):

    candidate_rows = []

    # --------------------------------------------------------
    # Select rule ONLY on fit portion
    # --------------------------------------------------------

    for policy in source_policies:

        for threshold in thresholds:

            result = policy_score(
                fit_idx,
                threshold,
                policy,
            )

            candidate_rows.append({
                "threshold":
                    threshold,

                "allowed_sources":
                    "+".join(policy),

                **result,
            })


    candidate_df = pd.DataFrame(
        candidate_rows
    )

    # Primary objective = macro F1
    # Secondary = balanced accuracy
    # Tertiary = fewer overrides
    candidate_df = (
        candidate_df
        .sort_values(
            [
                "macro_f1",
                "balanced_accuracy",
                "overrides",
            ],
            ascending=[
                False,
                False,
                True,
            ],
        )
        .reset_index(drop=True)
    )

    best = candidate_df.iloc[0]

    best_threshold = float(
        best["threshold"]
    )

    best_sources = (
        best["allowed_sources"]
        .split("+")
    )


    # --------------------------------------------------------
    # Apply selected rule to untouched eval fold
    # --------------------------------------------------------

    eval_pred, eval_accept = apply_policy(
        eval_idx,
        best_threshold,
        best_sources,
    )

    crossfit_pred[
        eval_idx
    ] = eval_pred


    eval_metrics = {
        "accuracy":
            accuracy_score(
                y_train[eval_idx],
                eval_pred,
            ),

        "balanced_accuracy":
            balanced_accuracy_score(
                y_train[eval_idx],
                eval_pred,
            ),

        "macro_f1":
            f1_score(
                y_train[eval_idx],
                eval_pred,
                average="macro",
            ),

        "weighted_f1":
            f1_score(
                y_train[eval_idx],
                eval_pred,
                average="weighted",
            ),
    }


    selected_rules.append({
        "fold": fold,

        "threshold":
            best_threshold,

        "allowed_sources":
            "+".join(best_sources),

        "train_macro_f1":
            best["macro_f1"],

        "eval_overrides":
            int(eval_accept.sum()),

        **{
            f"eval_{k}": v
            for k, v
            in eval_metrics.items()
        },
    })


selected_rules_df = pd.DataFrame(
    selected_rules
)

display(
    selected_rules_df.round(4)
)

,fold,threshold,allowed_sources,train_macro_f1,eval_overrides,eval_accuracy,eval_balanced_accuracy,eval_macro_f1,eval_weighted_f1
0,1,0.325,workflow_error+tool_use_error,0.5175,9,0.4899,0.4384,0.4629,0.4911
1,2,0.300,workflow_error+tool_use_error,0.4991,6,0.5403,0.5393,0.5420,0.5375
2,3,0.300,workflow_error+tool_use_error,0.5051,5,0.5537,0.5652,0.5172,0.5503
3,4,0.300,workflow_error+tool_use_error,0.5150,7,0.5503,0.4483,0.4624,0.5492
4,5,0.350,workflow_error+tool_use_error+grounding_state_...,0.5035,10,0.5253,0.5152,0.5174,0.5242


In [70]:
# ============================================================
# 62. Cross-fitted correction performance
# ============================================================

crossfit_metrics = evaluate_preds(
    y_train,
    crossfit_pred,
)

semantic_metrics = evaluate_preds(
    y_train,
    oof_sem_pred,
)

transition_metrics = evaluate_preds(
    y_train,
    oof_trans_pred,
)

compact_metrics = evaluate_preds(
    y_train,
    compact_routed_pred,
)


comparison_crossfit = pd.DataFrame([
    {
        "model": "semantic",
        **semantic_metrics,
    },
    {
        "model": "transition",
        **transition_metrics,
    },
    {
        "model": "compact_router",
        **compact_metrics,
    },
    {
        "model":
            "crossfit_targeted_correction",
        **crossfit_metrics,
    },
])


display(
    comparison_crossfit
    .sort_values(
        "macro_f1",
        ascending=False,
    )
    .round(4)
)


# ------------------------------------------------------------
# Rescues / breaks
# ------------------------------------------------------------

semantic_correct = (
    oof_sem_pred == y_train
)

crossfit_correct = (
    crossfit_pred == y_train
)

rescues = (
    (~semantic_correct)
    &
    crossfit_correct
)

breaks = (
    semantic_correct
    &
    (~crossfit_correct)
)


print(
    "Cross-fitted rescues:",
    rescues.sum(),
)

print(
    "Cross-fitted breaks:",
    breaks.sum(),
)

print(
    "Net rescues:",
    rescues.sum() - breaks.sum(),
)

print(
    "Accuracy delta:",
    crossfit_metrics["accuracy"]
    -
    semantic_metrics["accuracy"],
)

,model,accuracy,balanced_accuracy,macro_f1,weighted_f1
3,crossfit_targeted_correction,0.5319,0.4988,0.5052,0.5319
2,compact_router,0.5353,0.4918,0.5008,0.5335
1,transition,0.5353,0.4839,0.4981,0.5323
0,semantic,0.5252,0.4908,0.4974,0.5231


Cross-fitted rescues: 20
Cross-fitted breaks: 10
Net rescues: 10
Accuracy delta: 0.00671591672263272


In [71]:
# ============================================================
# 63. How stable is the selected rule?
# ============================================================

print("Selected rules by fold:")
display(
    selected_rules_df[
        [
            "fold",
            "threshold",
            "allowed_sources",
            "train_macro_f1",
            "eval_macro_f1",
            "eval_overrides",
        ]
    ].round(4)
)

print("\nThreshold frequency:")
print(
    selected_rules_df[
        "threshold"
    ].value_counts()
)

print("\nSource-policy frequency:")
print(
    selected_rules_df[
        "allowed_sources"
    ].value_counts()
)

Selected rules by fold:


,fold,threshold,allowed_sources,train_macro_f1,eval_macro_f1,eval_overrides
0,1,0.325,workflow_error+tool_use_error,0.5175,0.4629,9
1,2,0.300,workflow_error+tool_use_error,0.4991,0.5420,6
2,3,0.300,workflow_error+tool_use_error,0.5051,0.5172,5
3,4,0.300,workflow_error+tool_use_error,0.5150,0.4624,7
4,5,0.350,workflow_error+tool_use_error+grounding_state_...,0.5035,0.5174,10



Threshold frequency:
threshold
0.300    3
0.325    1
0.350    1
Name: count, dtype: int64

Source-policy frequency:
allowed_sources
workflow_error+tool_use_error                          4
workflow_error+tool_use_error+grounding_state_error    1
Name: count, dtype: int64


In [72]:
# ============================================================
# 64. Build learned constraint-override dataset
# ============================================================

import numpy as np
import pandas as pd

# ------------------------------------------------------------
# Proposal population
#
# Semantic says something other than constraint_error,
# while trajectory/transition proposes constraint_error.
# ------------------------------------------------------------

constraint_id = class_names.index("constraint_error")

proposal_mask = (
    (oof_sem_pred != constraint_id)
    &
    (oof_trans_pred == constraint_id)
)

proposal_idx = np.where(proposal_mask)[0]

print("Total training examples:", len(y_train))
print("Constraint proposals:", len(proposal_idx))


# ------------------------------------------------------------
# Target:
# 1 = accepting transition fixes semantic
# 0 = accepting transition would be wrong
# ------------------------------------------------------------

override_target = (
    y_train[proposal_idx] == constraint_id
).astype(int)

print("Good overrides:", override_target.sum())
print("Bad overrides:", len(override_target) - override_target.sum())
print("Positive prevalence:", override_target.mean())

Total training examples: 1489
Constraint proposals: 56
Good overrides: 26
Bad overrides: 30
Positive prevalence: 0.4642857142857143


In [76]:
# ============================================================
# Find existing transition arrays / matrices
# ============================================================

candidates = []

for name, obj in list(globals().items()):

    if isinstance(obj, np.ndarray):

        if (
            obj.ndim == 2
            and obj.shape[0] == len(y_train)
        ):
            candidates.append(
                (name, obj.shape)
            )

candidates

[('train_current_embeddings', (1489, 384)),
 ('T_train', (1489, 16)),
 ('X_sem_train', (1489, 384)),
 ('X_trans_train', (1489, 400)),
 ('oof_sem_prob', (1489, 5)),
 ('oof_trans_prob', (1489, 5)),
 ('sem_sorted', (1489, 5)),
 ('trans_sorted', (1489, 5))]

In [77]:
# ============================================================
# 65. Compact override features using T_train directly
# ============================================================

eps = 1e-12

# ------------------------------------------------------------
# Semantic uncertainty
# ------------------------------------------------------------

sem_sorted = np.sort(
    oof_sem_prob,
    axis=1,
)

semantic_confidence = (
    oof_sem_prob.max(axis=1)
)

semantic_margin = (
    sem_sorted[:, -1]
    - sem_sorted[:, -2]
)

semantic_entropy = -np.sum(
    oof_sem_prob
    * np.log(oof_sem_prob + eps),
    axis=1,
)


# ------------------------------------------------------------
# Transition uncertainty
# ------------------------------------------------------------

trans_sorted = np.sort(
    oof_trans_prob,
    axis=1,
)

transition_confidence = (
    oof_trans_prob.max(axis=1)
)

transition_margin = (
    trans_sorted[:, -1]
    - trans_sorted[:, -2]
)

transition_entropy = -np.sum(
    oof_trans_prob
    * np.log(oof_trans_prob + eps),
    axis=1,
)


# ------------------------------------------------------------
# Constraint-specific probabilities
# ------------------------------------------------------------

semantic_constraint_prob = (
    oof_sem_prob[:, constraint_id]
)

transition_constraint_prob = (
    oof_trans_prob[:, constraint_id]
)

constraint_prob_gain = (
    transition_constraint_prob
    - semantic_constraint_prob
)


# ------------------------------------------------------------
# Verify transition feature matrix
# ------------------------------------------------------------

print("T_train:", T_train.shape)

print(
    "transition_names:",
    len(transition_names),
)

assert T_train.shape[0] == len(y_train)
assert T_train.shape[1] == len(transition_names)

print("\nTransition features:")
for name in transition_names:
    print(" ", name)

T_train: (1489, 16)
transition_names: 16

Transition features:
  current_tminus3_cosine
  current_tminus3_l1
  current_tminus3_l2
  tminus3_present
  current_tminus2_cosine
  current_tminus2_l1
  current_tminus2_l2
  tminus2_present
  current_tminus1_cosine
  current_tminus1_l1
  current_tminus1_l2
  tminus1_present
  history_transition_1_cosine
  history_transition_1_l2
  history_transition_2_cosine
  history_transition_2_l2


In [78]:
# ============================================================
# 66. Transition feature dataframe
# ============================================================

transition_feature_df = pd.DataFrame(
    T_train,
    columns=transition_names,
)

print(
    transition_feature_df.shape
)

display(
    transition_feature_df.head()
)

(1489, 16)


,current_tminus3_cosine,current_tminus3_l1,current_tminus3_l2,tminus3_present,current_tminus2_cosine,current_tminus2_l1,current_tminus2_l2,tminus2_present,current_tminus1_cosine,current_tminus1_l1,current_tminus1_l2,tminus1_present,history_transition_1_cosine,history_transition_1_l2,history_transition_2_cosine,history_transition_2_l2
0,0.000000,0.000000,0.000000,0.0,0.505129,0.040755,0.994857,1.0,0.526036,0.039915,0.973616,1.0,0.000000,0.000000,0.961313,0.278161
1,0.000000,0.000000,0.000000,0.0,0.000000,0.000000,0.000000,0.0,0.405556,0.043941,1.090362,1.0,0.000000,0.000000,0.000000,0.000000
2,0.000000,0.000000,0.000000,0.0,0.965396,0.010558,0.263075,1.0,0.973451,0.009315,0.230429,1.0,0.000000,0.000000,0.992547,0.122094
3,0.455158,0.042642,1.043879,1.0,0.486656,0.041678,1.013256,1.0,0.497407,0.041609,1.002589,1.0,0.992547,0.122094,0.973451,0.230429
4,0.659610,0.033107,0.825094,1.0,0.707023,0.030622,0.765477,1.0,0.662295,0.032973,0.821834,1.0,0.892029,0.464696,0.832603,0.578614


In [79]:
# ============================================================
# 67. Override proposal dataframe
# ============================================================

override_df = pd.DataFrame({
    "row_idx":
        proposal_idx,

    "target":
        override_target,

    "semantic_pred":
        oof_sem_pred[
            proposal_idx
        ],

    "semantic_confidence":
        semantic_confidence[
            proposal_idx
        ],

    "semantic_margin":
        semantic_margin[
            proposal_idx
        ],

    "semantic_entropy":
        semantic_entropy[
            proposal_idx
        ],

    "semantic_constraint_prob":
        semantic_constraint_prob[
            proposal_idx
        ],

    "transition_confidence":
        transition_confidence[
            proposal_idx
        ],

    "transition_margin":
        transition_margin[
            proposal_idx
        ],

    "transition_entropy":
        transition_entropy[
            proposal_idx
        ],

    "transition_constraint_prob":
        transition_constraint_prob[
            proposal_idx
        ],

    "constraint_prob_gain":
        constraint_prob_gain[
            proposal_idx
        ],
})

In [80]:
# ------------------------------------------------------------
# Add trajectory features from T_train
# ------------------------------------------------------------

wanted_transition_features = [
    "current_tminus1_cosine",
    "current_tminus2_cosine",
    "current_tminus3_cosine",
    "history_transition_1_cosine",
    "history_transition_2_cosine",
]

available_trajectory_features = [
    feature
    for feature in wanted_transition_features
    if feature in transition_feature_df.columns
]

print(
    "Using trajectory features:",
    available_trajectory_features
)


for feature in available_trajectory_features:

    override_df[feature] = (
        transition_feature_df.loc[
            proposal_idx,
            feature,
        ]
        .to_numpy()
    )

Using trajectory features: ['current_tminus1_cosine', 'current_tminus2_cosine', 'current_tminus3_cosine', 'history_transition_1_cosine', 'history_transition_2_cosine']


In [81]:
# ------------------------------------------------------------
# History length
# ------------------------------------------------------------

if "history_event_count" in train_targets.columns:

    override_df[
        "history_event_count"
    ] = (
        train_targets.loc[
            proposal_idx,
            "history_event_count",
        ]
        .to_numpy()
    )

print(
    "Override dataset:",
    override_df.shape
)

display(
    override_df.head()
)

Override dataset: (56, 18)


,row_idx,target,semantic_pred,semantic_confidence,semantic_margin,semantic_entropy,semantic_constraint_prob,transition_confidence,transition_margin,transition_entropy,transition_constraint_prob,constraint_prob_gain,current_tminus1_cosine,current_tminus2_cosine,current_tminus3_cosine,history_transition_1_cosine,history_transition_2_cosine,history_event_count
0,5,1,0,0.490638,0.134253,1.113564,0.356385,0.454329,0.066295,1.166643,0.454329,0.097944,0.00000,0.000000,0.000000,0.000000,0.000000,0
1,17,0,3,0.637456,0.430462,1.025261,0.206994,0.424874,0.046490,1.232305,0.424874,0.217880,0.00000,0.000000,0.000000,0.000000,0.000000,0
2,100,0,2,0.302994,0.008457,1.367128,0.294537,0.397761,0.176137,1.343432,0.397761,0.103224,0.00000,0.000000,0.000000,0.000000,0.000000,0
3,109,1,0,0.308925,0.016110,1.408996,0.292814,0.338595,0.084165,1.437535,0.338595,0.045781,0.30016,0.293448,0.081081,0.458086,0.998635,4
4,141,1,0,0.336486,0.002630,1.277721,0.333856,0.397172,0.051764,1.298323,0.397172,0.063316,0.19185,0.718754,0.149232,0.123007,0.087924,12


In [82]:
# ============================================================
# 68. Semantic source-family features
# ============================================================

semantic_source = np.array([
    class_names[int(i)]
    for i in oof_sem_pred[
        proposal_idx
    ]
])

override_df[
    "semantic_pred_name"
] = semantic_source


source_dummies = pd.get_dummies(
    override_df[
        "semantic_pred_name"
    ],
    prefix="source",
    dtype=float,
)

override_df = pd.concat(
    [
        override_df,
        source_dummies,
    ],
    axis=1,
)

display(
    override_df
    .groupby(
        "semantic_pred_name"
    )["target"]
    .agg(
        count="size",
        good_overrides="sum",
        override_precision="mean",
    )
    .sort_values(
        "override_precision",
        ascending=False,
    )
    .round(4)
)

,count,good_overrides,override_precision
semantic_pred_name,,,
tool_use_error,11,7,0.6364
workflow_error,28,15,0.5357
grounding_state_error,17,4,0.2353


In [83]:
# ============================================================
# 69. Learned override feature matrix
# ============================================================

base_features = [
    "semantic_confidence",
    "semantic_margin",
    "semantic_entropy",
    "semantic_constraint_prob",

    "transition_confidence",
    "transition_margin",
    "transition_entropy",
    "transition_constraint_prob",

    "constraint_prob_gain",
]


base_features += (
    available_trajectory_features
)


if (
    "history_event_count"
    in override_df.columns
):
    base_features.append(
        "history_event_count"
    )


dummy_features = [
    column
    for column in override_df.columns
    if column.startswith(
        "source_"
    )
]


override_features = (
    base_features
    + dummy_features
)


X_override = (
    override_df[
        override_features
    ]
    .astype(float)
    .to_numpy()
)

y_override = (
    override_df[
        "target"
    ]
    .to_numpy()
    .astype(int)
)


print(
    "X_override:",
    X_override.shape
)

print(
    "y_override:",
    y_override.shape
)

print(
    "Good:",
    y_override.sum()
)

print(
    "Bad:",
    (
        y_override == 0
    ).sum()
)

print("\nFeatures:")

for feature in override_features:
    print(" ", feature)

X_override: (56, 18)
y_override: (56,)
Good: 26
Bad: 30

Features:
  semantic_confidence
  semantic_margin
  semantic_entropy
  semantic_constraint_prob
  transition_confidence
  transition_margin
  transition_entropy
  transition_constraint_prob
  constraint_prob_gain
  current_tminus1_cosine
  current_tminus2_cosine
  current_tminus3_cosine
  history_transition_1_cosine
  history_transition_2_cosine
  history_event_count
  source_grounding_state_error
  source_tool_use_error
  source_workflow_error


In [84]:
# ============================================================
# 70. Cross-fitted learned override probabilities
# ============================================================

from sklearn.model_selection import StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    average_precision_score,
    roc_auc_score,
)

C_grid = [
    0.01,
    0.03,
    0.1,
    0.3,
    1.0,
    3.0,
]

outer_cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42,
)

override_oof_prob = np.zeros(
    len(y_override),
    dtype=float,
)

selected_Cs = []


for fold, (train_idx, val_idx) in enumerate(
    outer_cv.split(
        X_override,
        y_override,
    ),
    start=1,
):

    X_outer_train = X_override[train_idx]
    y_outer_train = y_override[train_idx]

    # --------------------------------------------------------
    # Inner CV chooses C using PR-AUC
    # --------------------------------------------------------

    minority_count = np.bincount(
        y_outer_train
    ).min()

    inner_splits = min(
        4,
        int(minority_count),
    )

    inner_cv = StratifiedKFold(
        n_splits=inner_splits,
        shuffle=True,
        random_state=100 + fold,
    )

    C_scores = []

    for C in C_grid:

        fold_scores = []

        for inner_train_idx, inner_val_idx in inner_cv.split(
            X_outer_train,
            y_outer_train,
        ):

            model = Pipeline([
                (
                    "scale",
                    StandardScaler(),
                ),
                (
                    "clf",
                    LogisticRegression(
                        C=C,
                        class_weight="balanced",
                        max_iter=5000,
                        random_state=42,
                    ),
                ),
            ])

            model.fit(
                X_outer_train[
                    inner_train_idx
                ],
                y_outer_train[
                    inner_train_idx
                ],
            )

            prob = model.predict_proba(
                X_outer_train[
                    inner_val_idx
                ]
            )[:, 1]

            score = average_precision_score(
                y_outer_train[
                    inner_val_idx
                ],
                prob,
            )

            fold_scores.append(
                score
            )

        C_scores.append({
            "C": C,
            "mean_pr_auc":
                np.mean(
                    fold_scores
                ),
        })


    C_scores_df = pd.DataFrame(
        C_scores
    )

    best_C = float(
        C_scores_df
        .sort_values(
            "mean_pr_auc",
            ascending=False,
        )
        .iloc[0]["C"]
    )

    selected_Cs.append(
        best_C
    )


    # --------------------------------------------------------
    # Fit outer-fold model
    # --------------------------------------------------------

    final_model = Pipeline([
        (
            "scale",
            StandardScaler(),
        ),
        (
            "clf",
            LogisticRegression(
                C=best_C,
                class_weight="balanced",
                max_iter=5000,
                random_state=42,
            ),
        ),
    ])

    final_model.fit(
        X_override[
            train_idx
        ],
        y_override[
            train_idx
        ],
    )

    override_oof_prob[
        val_idx
    ] = (
        final_model
        .predict_proba(
            X_override[
                val_idx
            ]
        )[:, 1]
    )

    print(
        f"Fold {fold} | "
        f"selected C={best_C}"
    )


print(
    "\nSelected C values:",
    selected_Cs,
)

print(
    "\nPositive prevalence:",
    y_override.mean(),
)

print(
    "OOF PR-AUC:",
    average_precision_score(
        y_override,
        override_oof_prob,
    ),
)

print(
    "OOF ROC-AUC:",
    roc_auc_score(
        y_override,
        override_oof_prob,
    ),
)

Fold 1 | selected C=0.03
Fold 2 | selected C=0.01
Fold 3 | selected C=0.01
Fold 4 | selected C=0.03
Fold 5 | selected C=0.01

Selected C values: [0.03, 0.01, 0.01, 0.03, 0.01]

Positive prevalence: 0.4642857142857143
OOF PR-AUC: 0.6406988810646207
OOF ROC-AUC: 0.7064102564102565


In [85]:
# ============================================================
# 71. Learned override score separation
# ============================================================

override_df[
    "learned_override_prob"
] = (
    override_oof_prob
)


prob_summary = (
    override_df
    .groupby(
        "target"
    )[
        "learned_override_prob"
    ]
    .agg(
        [
            "count",
            "mean",
            "median",
            "std",
            "min",
            "max",
        ]
    )
)

display(
    prob_summary.round(4)
)

,count,mean,median,std,min,max
target,,,,,,
0,30,0.4700,0.4545,0.1061,0.2741,0.7955
1,26,0.5225,0.5436,0.0949,0.3097,0.7200


In [86]:
# ============================================================
# 72. Learned override intervention curve
# ============================================================

learned_rows = []

for threshold in np.arange(
    0.20,
    0.81,
    0.05,
):

    accept_local = (
        override_oof_prob
        >= threshold
    )

    accepted_count = int(
        accept_local.sum()
    )

    good = int(
        (
            accept_local
            &
            (y_override == 1)
        ).sum()
    )

    bad = int(
        (
            accept_local
            &
            (y_override == 0)
        ).sum()
    )

    corrected_pred = (
        oof_sem_pred.copy()
    )

    corrected_pred[
        proposal_idx[
            accept_local
        ]
    ] = constraint_id

    metrics = evaluate_preds(
        y_train,
        corrected_pred,
    )

    learned_rows.append({
        "threshold":
            threshold,

        "overrides":
            accepted_count,

        "good":
            good,

        "bad":
            bad,

        "net":
            good - bad,

        "precision":
            (
                good / accepted_count
                if accepted_count > 0
                else np.nan
            ),

        **metrics,
    })


learned_override_curve = pd.DataFrame(
    learned_rows
)


display(
    learned_override_curve
    .sort_values(
        [
            "macro_f1",
            "balanced_accuracy",
        ],
        ascending=False,
    )
    .round(4)
)

,threshold,overrides,good,bad,net,precision,accuracy,balanced_accuracy,macro_f1,weighted_f1
5,0.45,37,21,16,5,0.5676,0.5339,0.5005,0.5072,0.5338
3,0.35,53,25,28,-3,0.4717,0.5332,0.4989,0.5061,0.5334
2,0.30,55,26,29,-3,0.4727,0.5332,0.4988,0.5060,0.5333
0,0.20,56,26,30,-4,0.4643,0.5326,0.4985,0.5056,0.5327
1,0.25,56,26,30,-4,0.4643,0.5326,0.4985,0.5056,0.5327
4,0.40,47,23,24,-1,0.4894,0.5326,0.4985,0.5055,0.5325
6,0.50,26,17,9,8,0.6538,0.5326,0.4986,0.5053,0.5319
7,0.55,15,12,3,9,0.8000,0.5319,0.4977,0.5044,0.5308
8,0.60,8,5,3,2,0.6250,0.5272,0.4933,0.5000,0.5257
9,0.65,3,1,2,-1,0.3333,0.5252,0.4911,0.4977,0.5233


In [87]:
# ============================================================
# 73. Learned selector vs validated simple rule
# ============================================================

best_learned_row = (
    learned_override_curve
    .sort_values(
        [
            "macro_f1",
            "balanced_accuracy",
        ],
        ascending=False,
    )
    .iloc[0]
)

print(
    "Best learned selector:"
)

print(
    best_learned_row
)


comparison = pd.DataFrame([
    {
        "model":
            "semantic",
        **evaluate_preds(
            y_train,
            oof_sem_pred,
        ),
    },

    {
        "model":
            "transition",
        **evaluate_preds(
            y_train,
            oof_trans_pred,
        ),
    },

    {
        "model":
            "compact_router",
        **evaluate_preds(
            y_train,
            compact_routed_pred,
        ),
    },

    {
        "model":
            "crossfit_targeted_rule",
        **crossfit_metrics,
    },

    {
        "model":
            "learned_constraint_selector",
        "accuracy":
            best_learned_row[
                "accuracy"
            ],
        "balanced_accuracy":
            best_learned_row[
                "balanced_accuracy"
            ],
        "macro_f1":
            best_learned_row[
                "macro_f1"
            ],
        "weighted_f1":
            best_learned_row[
                "weighted_f1"
            ],
    },
])


display(
    comparison
    .sort_values(
        "macro_f1",
        ascending=False,
    )
    .round(4)
)

Best learned selector:
threshold             0.450000
overrides            37.000000
good                 21.000000
bad                  16.000000
net                   5.000000
precision             0.567568
accuracy              0.533915
balanced_accuracy     0.500524
macro_f1              0.507165
weighted_f1           0.533762
Name: 5, dtype: float64


,model,accuracy,balanced_accuracy,macro_f1,weighted_f1
4,learned_constraint_selector,0.5339,0.5005,0.5072,0.5338
3,crossfit_targeted_rule,0.5319,0.4988,0.5052,0.5319
2,compact_router,0.5353,0.4918,0.5008,0.5335
1,transition,0.5353,0.4839,0.4981,0.5323
0,semantic,0.5252,0.4908,0.4974,0.5231


In [88]:
# ============================================================
# 74. Inspect learned override feature importance
# ============================================================

coef_rows = []

outer_cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42,
)

for fold, (train_idx, val_idx) in enumerate(
    outer_cv.split(
        X_override,
        y_override,
    ),
    start=1,
):

    selected_C = selected_Cs[
        fold - 1
    ]

    model = Pipeline([
        (
            "scale",
            StandardScaler(),
        ),
        (
            "clf",
            LogisticRegression(
                C=selected_C,
                class_weight="balanced",
                max_iter=5000,
                random_state=42,
            ),
        ),
    ])

    model.fit(
        X_override[train_idx],
        y_override[train_idx],
    )

    coef = (
        model.named_steps[
            "clf"
        ]
        .coef_[0]
    )

    for feature, value in zip(
        override_features,
        coef,
    ):

        coef_rows.append({
            "fold": fold,
            "feature": feature,
            "coefficient": value,
        })


coef_df = pd.DataFrame(
    coef_rows
)

coef_summary = (
    coef_df
    .groupby("feature")
    .agg(
        mean_coef=(
            "coefficient",
            "mean",
        ),
        median_coef=(
            "coefficient",
            "median",
        ),
        std_coef=(
            "coefficient",
            "std",
        ),
        min_coef=(
            "coefficient",
            "min",
        ),
        max_coef=(
            "coefficient",
            "max",
        ),
    )
)

coef_summary[
    "sign_consistency"
] = (
    coef_df
    .assign(
        positive=lambda x:
            x[
                "coefficient"
            ] > 0
    )
    .groupby("feature")[
        "positive"
    ]
    .mean()
)

display(
    coef_summary
    .assign(
        abs_mean=lambda x:
            x[
                "mean_coef"
            ].abs()
    )
    .sort_values(
        "abs_mean",
        ascending=False,
    )
    .drop(
        columns="abs_mean"
    )
    .round(4)
)

,mean_coef,median_coef,std_coef,min_coef,max_coef,sign_consistency
feature,,,,,,
transition_constraint_prob,0.0854,0.0636,0.0444,0.0478,0.1544,1.0
transition_confidence,0.0854,0.0636,0.0444,0.0478,0.1544,1.0
transition_margin,0.0831,0.0637,0.0502,0.0458,0.1682,1.0
source_grounding_state_error,-0.0765,-0.0610,0.0334,-0.1227,-0.0475,0.0
transition_entropy,-0.0612,-0.0395,0.0353,-0.1103,-0.0306,0.0
current_tminus1_cosine,0.0575,0.0493,0.0332,0.0241,0.0977,1.0
source_tool_use_error,0.0559,0.0448,0.0347,0.0202,0.0947,1.0
semantic_constraint_prob,0.0428,0.0337,0.0276,0.0205,0.0872,1.0
current_tminus2_cosine,0.0425,0.0297,0.0391,0.0027,0.1020,1.0


In [89]:
# ============================================================
# 75. Feature-family ablation experiment
# ============================================================

feature_sets = {
    "probabilities_only": [
        "semantic_confidence",
        "semantic_margin",
        "semantic_entropy",
        "semantic_constraint_prob",
        "transition_confidence",
        "transition_margin",
        "transition_entropy",
        "transition_constraint_prob",
        "constraint_prob_gain",
    ],

    "trajectory_only": (
        available_trajectory_features
        + (
            ["history_event_count"]
            if "history_event_count"
            in override_df.columns
            else []
        )
    ),

    "source_only":
        dummy_features,

    "probabilities_plus_source": (
        [
            "semantic_confidence",
            "semantic_margin",
            "semantic_entropy",
            "semantic_constraint_prob",
            "transition_confidence",
            "transition_margin",
            "transition_entropy",
            "transition_constraint_prob",
            "constraint_prob_gain",
        ]
        + dummy_features
    ),

    "probabilities_plus_trajectory": (
        [
            "semantic_confidence",
            "semantic_margin",
            "semantic_entropy",
            "semantic_constraint_prob",
            "transition_confidence",
            "transition_margin",
            "transition_entropy",
            "transition_constraint_prob",
            "constraint_prob_gain",
        ]
        + available_trajectory_features
        + (
            ["history_event_count"]
            if "history_event_count"
            in override_df.columns
            else []
        )
    ),

    "all_features":
        override_features,
}

In [90]:
# ============================================================
# 76. Cross-fitted ablation PR-AUC
# ============================================================

ablation_rows = []

for feature_set_name, features in feature_sets.items():

    X = (
        override_df[
            features
        ]
        .astype(float)
        .to_numpy()
    )

    prob = np.zeros(
        len(y_override),
        dtype=float,
    )

    cv = StratifiedKFold(
        n_splits=5,
        shuffle=True,
        random_state=42,
    )

    for tr_idx, va_idx in cv.split(
        X,
        y_override,
    ):

        model = Pipeline([
            (
                "scale",
                StandardScaler(),
            ),
            (
                "clf",
                LogisticRegression(
                    C=0.03,
                    class_weight="balanced",
                    max_iter=5000,
                    random_state=42,
                ),
            ),
        ])

        model.fit(
            X[tr_idx],
            y_override[tr_idx],
        )

        prob[va_idx] = (
            model.predict_proba(
                X[va_idx]
            )[:, 1]
        )


    ablation_rows.append({
        "feature_set":
            feature_set_name,

        "n_features":
            len(features),

        "pr_auc":
            average_precision_score(
                y_override,
                prob,
            ),

        "roc_auc":
            roc_auc_score(
                y_override,
                prob,
            ),
    })


ablation_df = pd.DataFrame(
    ablation_rows
)

display(
    ablation_df
    .sort_values(
        "pr_auc",
        ascending=False,
    )
    .round(4)
)

,feature_set,n_features,pr_auc,roc_auc
3,probabilities_plus_source,12,0.6349,0.7090
0,probabilities_only,9,0.6294,0.6974
5,all_features,18,0.6218,0.6974
4,probabilities_plus_trajectory,15,0.5984,0.6667
2,source_only,3,0.5139,0.5955
1,trajectory_only,6,0.4779,0.5096


In [91]:
# ============================================================
# 77. Expert probability-disagreement features
# ============================================================

import numpy as np
import pandas as pd

eps = 1e-10

P_sem = np.clip(
    oof_sem_prob.copy(),
    eps,
    1.0,
)

P_trans = np.clip(
    oof_trans_prob.copy(),
    eps,
    1.0,
)

# Normalize defensively
P_sem = P_sem / P_sem.sum(axis=1, keepdims=True)
P_trans = P_trans / P_trans.sum(axis=1, keepdims=True)


# ------------------------------------------------------------
# Distribution distances
# ------------------------------------------------------------

prob_l1 = np.abs(
    P_sem - P_trans
).sum(axis=1)

prob_l2 = np.sqrt(
    ((P_sem - P_trans) ** 2).sum(axis=1)
)

prob_dot = (
    P_sem * P_trans
).sum(axis=1)

prob_cosine = prob_dot / (
    np.linalg.norm(P_sem, axis=1)
    * np.linalg.norm(P_trans, axis=1)
    + eps
)


# ------------------------------------------------------------
# KL / Jensen-Shannon
# ------------------------------------------------------------

kl_sem_to_trans = (
    P_sem
    * np.log(
        P_sem / P_trans
    )
).sum(axis=1)

kl_trans_to_sem = (
    P_trans
    * np.log(
        P_trans / P_sem
    )
).sum(axis=1)

M = 0.5 * (
    P_sem + P_trans
)

js_divergence = 0.5 * (
    (
        P_sem
        * np.log(P_sem / M)
    ).sum(axis=1)
    +
    (
        P_trans
        * np.log(P_trans / M)
    ).sum(axis=1)
)


# ------------------------------------------------------------
# Cross-expert probabilities
# ------------------------------------------------------------

rows = np.arange(
    len(P_sem)
)

semantic_class_under_transition = (
    P_trans[
        rows,
        oof_sem_pred
    ]
)

transition_class_under_semantic = (
    P_sem[
        rows,
        oof_trans_pred
    ]
)

semantic_own_prob = (
    P_sem[
        rows,
        oof_sem_pred
    ]
)

transition_own_prob = (
    P_trans[
        rows,
        oof_trans_pred
    ]
)


# How much more strongly does transition support its choice
# than semantic supports that same class?
transition_choice_gain = (
    transition_own_prob
    - transition_class_under_semantic
)

# How much does transition reduce support for semantic's choice?
semantic_choice_drop = (
    semantic_own_prob
    - semantic_class_under_transition
)


# ------------------------------------------------------------
# Cross-expert ranks
# ------------------------------------------------------------

sem_order = np.argsort(
    -P_sem,
    axis=1,
)

trans_order = np.argsort(
    -P_trans,
    axis=1,
)

transition_class_rank_under_semantic = np.array([
    np.where(
        sem_order[i] == oof_trans_pred[i]
    )[0][0] + 1
    for i in range(len(P_sem))
])

semantic_class_rank_under_transition = np.array([
    np.where(
        trans_order[i] == oof_sem_pred[i]
    )[0][0] + 1
    for i in range(len(P_sem))
])


# ------------------------------------------------------------
# Assemble
# ------------------------------------------------------------

expert_disagreement_df = pd.DataFrame({
    "prob_l1":
        prob_l1,

    "prob_l2":
        prob_l2,

    "prob_cosine":
        prob_cosine,

    "js_divergence":
        js_divergence,

    "kl_sem_to_trans":
        kl_sem_to_trans,

    "kl_trans_to_sem":
        kl_trans_to_sem,

    "semantic_class_under_transition":
        semantic_class_under_transition,

    "transition_class_under_semantic":
        transition_class_under_semantic,

    "transition_choice_gain":
        transition_choice_gain,

    "semantic_choice_drop":
        semantic_choice_drop,

    "transition_class_rank_under_semantic":
        transition_class_rank_under_semantic,

    "semantic_class_rank_under_transition":
        semantic_class_rank_under_transition,
})


display(
    expert_disagreement_df
    .describe()
    .T
    .round(4)
)

,count,mean,std,min,25%,50%,75%,max
prob_l1,1489.0,0.1913,0.1414,0.0046,0.0813,0.1623,0.2641,0.8116
prob_l2,1489.0,0.1142,0.0865,0.0021,0.0479,0.0959,0.1571,0.5009
prob_cosine,1489.0,0.9795,0.0345,0.6694,0.9774,0.9932,0.9989,1.0000
js_divergence,1489.0,0.0107,0.0126,0.0001,0.0025,0.0064,0.0140,0.1084
kl_sem_to_trans,1489.0,0.0409,0.0492,0.0002,0.0094,0.0240,0.0533,0.4468
kl_trans_to_sem,1489.0,0.0471,0.0572,0.0002,0.0107,0.0283,0.0600,0.6387
semantic_class_under_transition,1489.0,0.6183,0.2043,0.0896,0.4640,0.5953,0.7887,0.9845
transition_class_under_semantic,1489.0,0.6609,0.2110,0.0583,0.5007,0.6655,0.8537,0.9980
transition_choice_gain,1489.0,-0.0285,0.0967,-0.3703,-0.0851,-0.0283,0.0218,0.3553
semantic_choice_drop,1489.0,0.0600,0.0891,-0.2689,-0.0016,0.0506,0.1115,0.4058


In [92]:
# ============================================================
# 78. Attach disagreement features to override candidates
# ============================================================

# override_df already contains row_idx from the previous cell

disagreement_features = list(
    expert_disagreement_df.columns
)

for feature in disagreement_features:

    override_df[feature] = (
        expert_disagreement_df.loc[
            override_df["row_idx"].values,
            feature
        ].values
    )


print(
    "Override rows:",
    len(override_df)
)

display(
    override_df[
        disagreement_features
    ]
    .describe()
    .T
    .round(4)
)

Override rows: 56


,count,mean,std,min,25%,50%,75%,max
prob_l1,56.0,0.3074,0.1511,0.0877,0.1949,0.2493,0.3879,0.7829
prob_l2,56.0,0.1829,0.0918,0.0536,0.1170,0.1515,0.2322,0.4648
prob_cosine,56.0,0.9440,0.0529,0.7917,0.9211,0.9657,0.9811,0.9956
js_divergence,56.0,0.0189,0.0172,0.0022,0.0071,0.0128,0.0237,0.0862
kl_sem_to_trans,56.0,0.0751,0.0691,0.0088,0.0285,0.0499,0.0918,0.3364
kl_trans_to_sem,56.0,0.0802,0.0754,0.0088,0.0295,0.0556,0.0999,0.3866
semantic_class_under_transition,56.0,0.3197,0.0610,0.1518,0.2786,0.3268,0.3605,0.4308
transition_class_under_semantic,56.0,0.3214,0.0867,0.1567,0.2548,0.3195,0.3822,0.4626
transition_choice_gain,56.0,0.0840,0.0613,-0.0220,0.0347,0.0795,0.1205,0.2217
semantic_choice_drop,56.0,0.1393,0.0808,-0.0089,0.0858,0.1161,0.1846,0.3914


In [93]:
# ============================================================
# 79. Good vs bad override disagreement analysis
# ============================================================

disagreement_comparison = []

for feature in disagreement_features:

    good = override_df.loc[
        override_df["target"] == 1,
        feature
    ]

    bad = override_df.loc[
        override_df["target"] == 0,
        feature
    ]

    pooled_std = np.sqrt(
        (
            good.var(ddof=1)
            + bad.var(ddof=1)
        ) / 2
    )

    if pooled_std > 0:
        cohens_d = (
            good.mean()
            - bad.mean()
        ) / pooled_std
    else:
        cohens_d = np.nan

    disagreement_comparison.append({
        "feature": feature,
        "good_mean": good.mean(),
        "bad_mean": bad.mean(),
        "difference":
            good.mean() - bad.mean(),
        "cohens_d": cohens_d,
    })


disagreement_comparison = pd.DataFrame(
    disagreement_comparison
)

display(
    disagreement_comparison
    .assign(
        abs_d=lambda x:
            x["cohens_d"].abs()
    )
    .sort_values(
        "abs_d",
        ascending=False
    )
    .drop(columns="abs_d")
    .round(4)
)

,feature,good_mean,bad_mean,difference,cohens_d
7,transition_class_under_semantic,0.3432,0.3025,0.0408,0.4795
10,transition_class_rank_under_semantic,2.1154,2.3000,-0.1846,-0.4168
5,kl_trans_to_sem,0.0868,0.0744,0.0124,0.1611
3,js_divergence,0.0202,0.0178,0.0024,0.1386
4,kl_sem_to_trans,0.0802,0.0707,0.0095,0.1354
6,semantic_class_under_transition,0.3240,0.3159,0.0080,0.1298
8,transition_choice_gain,0.0880,0.0805,0.0075,0.1226
0,prob_l1,0.3167,0.2994,0.0173,0.1135
1,prob_l2,0.1877,0.1787,0.0090,0.0972
2,prob_cosine,0.9426,0.9451,-0.0025,-0.0458


In [94]:
# ============================================================
# 80. Compact probability-space selector
# ============================================================

probspace_features = [
    # original probability features
    "semantic_confidence",
    "semantic_margin",
    "semantic_entropy",
    "semantic_constraint_prob",

    "transition_confidence",
    "transition_margin",
    "transition_entropy",
    "transition_constraint_prob",

    "constraint_prob_gain",

    # cross-expert probability geometry
    "transition_class_under_semantic",
    "semantic_class_under_transition",
    "transition_choice_gain",
    "semantic_choice_drop",

    "transition_class_rank_under_semantic",
    "semantic_class_rank_under_transition",

    # source-family identity
    "source_grounding_state_error",
    "source_tool_use_error",
    "source_workflow_error",
]


X_probspace = (
    override_df[
        probspace_features
    ]
    .astype(float)
    .to_numpy()
)

y_probspace = (
    override_df[
        "target"
    ]
    .to_numpy()
    .astype(int)
)

print("X_probspace:", X_probspace.shape)
print("y:", y_probspace.shape)

X_probspace: (56, 18)
y: (56,)


In [95]:
# ============================================================
# 81. Cross-fitted PR-AUC comparison
# ============================================================

from sklearn.model_selection import StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    average_precision_score,
    roc_auc_score,
)

feature_versions = {
    "previous_all_18":
        X_override,

    "probability_space":
        X_probspace,
}

C_grid = [
    0.01,
    0.03,
    0.1,
    0.3,
    1.0,
]

selector_results = {}
selector_probs = {}


for model_name, X in feature_versions.items():

    outer_cv = StratifiedKFold(
        n_splits=5,
        shuffle=True,
        random_state=42,
    )

    oof_prob = np.zeros(
        len(y_probspace),
        dtype=float,
    )

    selected = []

    for fold, (tr_idx, va_idx) in enumerate(
        outer_cv.split(
            X,
            y_probspace,
        ),
        start=1,
    ):

        # --------------------------------------------
        # Inner C selection
        # --------------------------------------------

        X_tr = X[tr_idx]
        y_tr = y_probspace[tr_idx]

        inner_cv = StratifiedKFold(
            n_splits=4,
            shuffle=True,
            random_state=100 + fold,
        )

        C_rows = []

        for C in C_grid:

            inner_scores = []

            for itr, iva in inner_cv.split(
                X_tr,
                y_tr,
            ):

                model = Pipeline([
                    (
                        "scale",
                        StandardScaler(),
                    ),
                    (
                        "clf",
                        LogisticRegression(
                            C=C,
                            class_weight="balanced",
                            max_iter=5000,
                            random_state=42,
                        ),
                    ),
                ])

                model.fit(
                    X_tr[itr],
                    y_tr[itr],
                )

                p = model.predict_proba(
                    X_tr[iva]
                )[:, 1]

                inner_scores.append(
                    average_precision_score(
                        y_tr[iva],
                        p,
                    )
                )

            C_rows.append({
                "C": C,
                "score":
                    np.mean(inner_scores),
            })

        best_C = (
            pd.DataFrame(C_rows)
            .sort_values(
                "score",
                ascending=False,
            )
            .iloc[0]["C"]
        )

        selected.append(
            best_C
        )

        # --------------------------------------------
        # Outer prediction
        # --------------------------------------------

        model = Pipeline([
            (
                "scale",
                StandardScaler(),
            ),
            (
                "clf",
                LogisticRegression(
                    C=best_C,
                    class_weight="balanced",
                    max_iter=5000,
                    random_state=42,
                ),
            ),
        ])

        model.fit(
            X[tr_idx],
            y_probspace[tr_idx],
        )

        oof_prob[va_idx] = (
            model.predict_proba(
                X[va_idx]
            )[:, 1]
        )

    selector_probs[
        model_name
    ] = oof_prob

    selector_results[
        model_name
    ] = {
        "selected_Cs":
            selected,

        "pr_auc":
            average_precision_score(
                y_probspace,
                oof_prob,
            ),

        "roc_auc":
            roc_auc_score(
                y_probspace,
                oof_prob,
            ),
    }


selector_results

{'previous_all_18': {'selected_Cs': [0.03, 0.01, 0.01, 0.03, 0.01],
  'pr_auc': 0.6406988810646207,
  'roc_auc': 0.7064102564102565},
 'probability_space': {'selected_Cs': [0.03, 0.01, 0.01, 0.01, 0.01],
  'pr_auc': 0.6193760782135354,
  'roc_auc': 0.6897435897435898}}

In [96]:
# ============================================================
# 82. Probability-space selector intervention curve
# ============================================================

probspace_oof_prob = (
    selector_probs[
        "probability_space"
    ]
)

rows = []

for threshold in np.arange(
    0.30,
    0.76,
    0.05,
):

    accept = (
        probspace_oof_prob
        >= threshold
    )

    corrected = (
        oof_sem_pred.copy()
    )

    corrected[
        proposal_idx[
            accept
        ]
    ] = constraint_id

    good = int(
        (
            accept
            &
            (y_override == 1)
        ).sum()
    )

    bad = int(
        (
            accept
            &
            (y_override == 0)
        ).sum()
    )

    metrics = evaluate_preds(
        y_train,
        corrected,
    )

    rows.append({
        "threshold":
            threshold,

        "overrides":
            int(accept.sum()),

        "good":
            good,

        "bad":
            bad,

        "net":
            good - bad,

        "precision":
            (
                good / accept.sum()
                if accept.sum() > 0
                else np.nan
            ),

        **metrics,
    })


probspace_curve = pd.DataFrame(
    rows
)

display(
    probspace_curve
    .sort_values(
        [
            "macro_f1",
            "balanced_accuracy",
        ],
        ascending=False,
    )
    .round(4)
)

,threshold,overrides,good,bad,net,precision,accuracy,balanced_accuracy,macro_f1,weighted_f1
3,0.45,39,21,18,3,0.5385,0.5332,0.4997,0.5064,0.5332
4,0.50,25,17,8,9,0.6800,0.5332,0.4989,0.5056,0.5326
2,0.40,48,23,25,-2,0.4792,0.5326,0.4985,0.5054,0.5327
0,0.30,53,24,29,-5,0.4528,0.5319,0.4975,0.5046,0.5320
1,0.35,52,24,28,-4,0.4615,0.5319,0.4975,0.5046,0.5320
5,0.55,16,12,4,8,0.7500,0.5312,0.4974,0.5039,0.5302
6,0.60,8,5,3,2,0.6250,0.5272,0.4933,0.5000,0.5257
9,0.75,1,0,1,-1,0.0000,0.5245,0.4905,0.4971,0.5225
8,0.70,2,0,2,-2,0.0000,0.5245,0.4905,0.4970,0.5226
7,0.65,3,0,3,-3,0.0000,0.5238,0.4901,0.4967,0.5220


# Conclusions — Trajectory Context Utility

## Research objective

This notebook investigated whether **trajectory/history context provides useful predictive information beyond the current-turn semantic representation** for failure-family classification.

The analysis compared two main experts:

- **Semantic expert** — predicts the failure family from the current-turn semantic embedding.
- **Transition expert** — augments the current-turn representation with compact trajectory/transition features derived from recent interaction history.

The notebook then investigated three progressively more specific questions:

1. Does trajectory context improve failure-family prediction overall?
2. Can we learn when to trust the transition expert instead of the semantic expert?
3. Are there particular failure families for which trajectory context is systematically useful?

---

## 1. Overall semantic vs. transition performance

Out-of-fold evaluation on the training set showed:

| Model | Accuracy | Balanced Accuracy | Macro-F1 | Weighted F1 |
|---|---:|---:|---:|---:|
| Semantic | 0.5252 | 0.4908 | 0.4974 | 0.5231 |
| Transition | 0.5353 | 0.4839 | 0.4981 | 0.5323 |

The transition representation therefore produced a small improvement in:

- accuracy: approximately **+1.0 percentage point**
- weighted F1: approximately **+0.9 percentage point**

but essentially no improvement in macro-F1 and a decrease in balanced accuracy.

This is the first important result:

> **Trajectory context contains useful information, but its benefit is not uniform across classes.**

The transition model is not simply a globally superior classifier.

---

## 2. Rescue/break analysis

Across the 1,489 OOF examples:

- both experts correct: **728**
- both experts wrong: **638**
- transition rescues semantic: **69**
- transition breaks a correct semantic prediction: **54**

Thus:

- net rescues = **+15**
- semantic accuracy = **0.5252**
- transition accuracy = **0.5353**

The two experts disagree only on a relatively small subset of the data, but those disagreements contain meaningful complementary information.

The theoretical oracle that always chooses the correct expert reaches:

**Oracle accuracy = 0.5715**

compared with:

**Semantic accuracy = 0.5252**

giving an available oracle gain of:

**+0.0463**

Therefore, the main opportunity is not replacing the semantic model with the transition model, but determining **when trajectory information should override the semantic prediction**.

---

## 3. Trajectory utility is strongly failure-family dependent

The most important result of the notebook is the family-level decomposition.

| Failure family | Support | Semantic recall | Transition recall | Δ recall | Rescues | Breaks | Net |
|---|---:|---:|---:|---:|---:|---:|---:|
| constraint_error | 317 | 0.3817 | 0.4416 | **+0.0599** | 26 | 7 | **+19** |
| workflow_error | 660 | 0.6455 | 0.6576 | **+0.0121** | 28 | 20 | **+8** |
| grounding_state_error | 244 | 0.4590 | 0.4467 | -0.0123 | 12 | 15 | -3 |
| tool_use_error | 237 | 0.4515 | 0.4219 | -0.0295 | 3 | 10 | -7 |
| reasoning_value_error | 31 | 0.5161 | 0.4516 | -0.0645 | 0 | 2 | -2 |

This reveals a clear asymmetric pattern.

### Beneficial trajectory families

Trajectory context is beneficial primarily for:

1. **constraint_error**
2. **workflow_error**

The strongest effect is for `constraint_error`:

- +5.99 percentage points recall
- 26 rescues
- only 7 breaks
- net +19 correct predictions

### Harmful trajectory families

Trajectory context is harmful overall for:

- `grounding_state_error`
- `tool_use_error`
- `reasoning_value_error`

Therefore:

> **Trajectory context should not be treated as universally beneficial. Its utility depends strongly on the underlying failure family.**

---

## 4. Constraint errors are the clearest trajectory-sensitive family

The strongest finding is the improvement for `constraint_error`.

The semantic model correctly identifies only:

**38.17%**

of constraint errors, while the transition model reaches:

**44.16%**

This is the largest positive class-level recall change in the experiment.

Trajectory context produces:

- 26 constraint rescues
- 7 constraint breaks
- net +19

This suggests that some constraint violations are difficult to infer from the current turn alone.

Instead, their identity depends on information distributed across previous interaction states.

A plausible interpretation is that trajectory context helps recover information such as:

- previously established requirements,
- previous user constraints,
- prior actions,
- interaction state,
- accumulated obligations,
- or whether the current action is consistent with what happened earlier.

The notebook does not yet establish which of these mechanisms is responsible, but it clearly localizes the strongest trajectory benefit to the constraint-error family.

---

## 5. Transition gains are concentrated in specific expert disagreements

The semantic and transition experts disagree on only **183 / 1489** OOF examples.

Among those disagreements:

- transition wins: **69**
- semantic wins: **114**

So blindly choosing transition on disagreement is not optimal.

However, disagreement structure is highly non-uniform.

Several semantic → transition prediction pairs showed substantially different transition win rates.

For example:

- `workflow_error → constraint_error` showed a strong transition advantage in several trajectory regimes.
- `tool_use_error → constraint_error` also showed relatively high transition win rates.
- transitions toward some other classes were substantially less reliable.

This reinforces the conclusion that trajectory utility is **conditional on both the semantic prediction and the proposed transition prediction**.

---

## 6. Recent trajectory similarity contains some routing signal

Several trajectory similarity variables differed between transition rescues and breaks.

Among the clearest signals were:

- `current_tminus2_cosine`
- `current_tminus1_cosine`
- recent similarity summaries
- some history-length interactions

For example, transition wins were more common in the **medium recent-similarity regime** than at very low similarity.

The transition win rate by similarity bin was approximately:

| Similarity regime | Transition win rate |
|---|---:|
| Low | 0.246 |
| Medium | **0.541** |
| High | 0.344 |

This is notable because trajectory utility is not monotonic.

Extremely high similarity does not necessarily imply that the transition model should be trusted.

Instead:

> **Trajectory context appears most useful when history is relevant but not simply redundant with the current state.**

This is consistent with the idea that useful trajectory information reflects a meaningful state transition rather than mere repetition.

---

## 7. A global learned router provides only modest gains

A learned router was trained to choose between the semantic and transition experts.

Its OOF performance was approximately:

| Model | Accuracy | Balanced Accuracy | Macro-F1 | Weighted F1 |
|---|---:|---:|---:|---:|
| Semantic | 0.5252 | 0.4908 | 0.4974 | 0.5231 |
| Transition | 0.5353 | 0.4839 | 0.4981 | 0.5323 |
| Compact router | 0.5353 | **0.4918** | **0.5008** | **0.5335** |

The compact router produced:

- 31 rescues
- 16 breaks
- net +15
- accuracy delta over semantic: approximately **+0.0101**

The oracle gain available over semantic was **0.0463**, while the compact router captured only **0.0101**.

Thus it captured approximately:

**21.7% of the available oracle gain.**

This means that expert selection is possible, but difficult.

The transition expert contains complementary information, but determining exactly when that information is correct remains noisy.

---

## 8. Test-set routing evidence was directionally positive but not conclusive

On the held-out test set, the router produced:

- semantic correct: **128**
- transition correct: **139**
- routed correct: **140**
- switches: **90**
- rescues: **28**
- breaks: **16**
- net rescues: **+12**

The resulting routed accuracy was:

**0.4878**

compared with semantic accuracy of approximately:

**0.4460**

The bootstrap estimate for router vs. semantic was:

- mean improvement: **+0.0417**
- 95% CI: approximately **[-0.0035, 0.0871]**
- bootstrap probability of non-positive improvement: **0.0398**

McNemar analysis gave:

- rescues: 28
- breaks: 16
- p ≈ **0.096**

Thus the test result is encouraging but not strong enough to claim a statistically secure improvement.

Relative to the transition expert itself, the router added almost nothing:

- router vs transition net gain: **+1**
- McNemar p = **1.0**

Therefore the evidence supports complementarity over the semantic baseline, but not a robust improvement over simply using the transition expert.

---

## 9. Targeted constraint correction is more promising than global routing

Because family-level analysis identified `constraint_error` as the clearest trajectory-sensitive class, the notebook investigated a more targeted strategy:

> Keep the semantic prediction by default, but selectively override it with `constraint_error` when the transition model provides sufficient evidence.

There were:

- 1,189 candidate non-constraint semantic predictions
- 196 true constraint errors among those candidates

The raw trajectory rule proposed 56 constraint overrides:

- 26 correct
- 30 incorrect

giving:

- precision = **0.4643**
- recall = **0.1327**
- F1 = **0.2063**

Despite relatively low precision, this increased overall accuracy from:

**0.5252 → 0.5326**

and macro-F1 from:

**0.4974 → 0.5056**

This demonstrated that even a narrow family-specific correction can improve the global classifier.

---

## 10. Conservative constraint overrides work better

Increasing the constraint evidence threshold substantially improved override precision.

At a threshold of **0.40**:

- overrides: 28
- good overrides: 19
- bad overrides: 9
- precision: **0.6786**
- accuracy: **0.5346**
- balanced accuracy: **0.5002**
- macro-F1: **0.5070**

At higher thresholds, precision increased further but coverage became too small.

This establishes an important precision/coverage tradeoff:

> **Trajectory-based correction is most useful as a conservative intervention rather than a broad replacement strategy.**

---

## 11. Source-family conditioning improves the correction rule

Constraint corrections were substantially more reliable when the semantic model originally predicted:

- `workflow_error`
- `tool_use_error`

They were much less reliable when the semantic source prediction was:

- `grounding_state_error`

Observed override precision by source included approximately:

- `tool_use_error` → constraint: **0.636**
- `workflow_error` → constraint: **0.536**
- `grounding_state_error` → constraint: **0.235**

A conditional rule restricting overrides primarily to `workflow_error` and `tool_use_error` therefore performed better than an unrestricted constraint override.

The best in-sample conditional rule used:

- threshold ≈ **0.30**
- allowed sources: `workflow_error + tool_use_error`

and obtained approximately:

- accuracy = **0.5346**
- balanced accuracy = **0.5017**
- macro-F1 = **0.5081**

This was one of the strongest targeted results in the notebook.

---

## 12. Cross-fitting confirms a smaller but persistent targeted benefit

Because selecting the correction rule on the same examples used for evaluation can overestimate performance, the targeted strategy was evaluated with cross-fitting.

Across five folds, selected thresholds were generally around:

**0.30–0.35**

and four of five folds selected:

`workflow_error + tool_use_error`

as the preferred source-family policy.

Cross-fitted results were:

| Model | Accuracy | Balanced Accuracy | Macro-F1 | Weighted F1 |
|---|---:|---:|---:|---:|
| Semantic | 0.5252 | 0.4908 | 0.4974 | 0.5231 |
| Transition | 0.5353 | 0.4839 | 0.4981 | 0.5323 |
| Compact router | 0.5353 | 0.4918 | 0.5008 | 0.5335 |
| Cross-fit targeted correction | 0.5319 | **0.4988** | **0.5052** | 0.5319 |

The cross-fitted correction produced:

- 20 rescues
- 10 breaks
- net +10
- accuracy gain ≈ **+0.0067**

The gain is small, but importantly it survives out-of-fold rule selection.

This provides stronger evidence that the family-specific trajectory effect is real rather than purely the result of threshold tuning.

---

## 13. A learned constraint selector can distinguish useful overrides

The next experiment trained a classifier specifically on the 56 proposed constraint overrides.

The target was:

> Will this proposed transition-to-constraint override actually correct the semantic prediction?

Among the 56 proposals:

- good overrides: 26
- bad overrides: 30
- positive prevalence: **0.4643**

Five-fold OOF evaluation of the learned selector achieved:

- PR-AUC = **0.6407**
- ROC-AUC = **0.7064**

Thus the selector contains meaningful predictive signal above the 46.4% positive baseline.

At threshold **0.45**:

- overrides selected: 37
- good: 21
- bad: 16
- precision: **0.5676**
- accuracy: **0.5339**
- balanced accuracy: **0.5005**
- macro-F1: **0.5072**

This slightly exceeded the cross-fitted hand-designed targeted rule in macro-F1.

However, the improvement remains small because only 56 override examples are available.

---

## 14. The learned selector mostly relies on expert posterior information

Coefficient analysis showed the most stable positive features included:

- transition constraint probability
- transition confidence
- transition margin
- source = `tool_use_error`
- recent trajectory similarity
- semantic constraint probability

Negative evidence included:

- source = `grounding_state_error`
- transition entropy
- semantic entropy

This is consistent with the earlier descriptive analysis.

The selector learns that a transition correction is more trustworthy when:

1. the transition expert strongly supports its proposed class,
2. its prediction is relatively confident,
3. the semantic source family is one for which trajectory corrections historically help,
4. and some recent trajectory alignment exists.

---

## 15. Feature ablation reveals that raw trajectory features add little to the selector

Feature-set ablation produced:

| Feature set | PR-AUC | ROC-AUC |
|---|---:|---:|
| probabilities + source | **0.6349** | **0.7090** |
| probabilities only | 0.6294 | 0.6974 |
| all features | 0.6218 | 0.6974 |
| probabilities + trajectory | 0.5984 | 0.6667 |
| source only | 0.5139 | 0.5955 |
| trajectory only | 0.4779 | 0.5096 |

This is one of the notebook's most informative negative results.

Raw trajectory variables alone are essentially useless for deciding whether a proposed constraint override is correct:

**ROC-AUC ≈ 0.51**

Adding trajectory variables to probability features also fails to improve selection.

By contrast, the strongest compact representation is:

> **expert posterior information + semantic source-family identity**

This suggests that the transition classifier has already transformed the useful trajectory information into its posterior distribution.

In other words:

> **Trajectory features matter for constructing the transition expert, but they are not independently useful for routing once the transition expert's posterior is known.**

---

## 16. Richer probability-disagreement geometry also fails to improve selection

The notebook additionally tested probability-space disagreement features including:

- L1 probability distance
- L2 probability distance
- probability cosine similarity
- Jensen-Shannon divergence
- forward/reverse KL divergence
- cross-expert class probabilities
- choice gains
- cross-expert class ranks

These quantities clearly characterize disagreement between the experts.

For example, the 56 constraint proposals showed substantially greater posterior divergence than the full dataset.

However, they did not meaningfully distinguish good from bad overrides.

Effect sizes between good and bad proposals were generally small.

The largest observed differences were still modest.

When these probability-space features were used for the selector:

- previous 18-feature selector PR-AUC = **0.6407**
- probability-space selector PR-AUC = **0.6194**

and:

- previous ROC-AUC = **0.7064**
- probability-space ROC-AUC = **0.6897**

Thus the richer probability geometry degraded rather than improved OOF discrimination.

Its best downstream operating point reached only approximately:

- accuracy = **0.5332**
- balanced accuracy = **0.4997**
- macro-F1 = **0.5064**

compared with macro-F1 **0.5072** for the previous selector.

Therefore:

> **The magnitude and geometry of expert disagreement are less informative than which class is being proposed, the expert posterior confidence, and the source failure family.**

---

# Overall conclusions

This notebook provides evidence for five main conclusions.

### 1. Trajectory context contains real but limited complementary signal

The transition expert improves accuracy relative to the semantic expert and rescues examples that cannot be solved from the current-turn representation alone.

However, the gain is modest and trajectory context can also break correct semantic predictions.

Therefore trajectory context should not be interpreted as a universally superior representation.

---

### 2. Trajectory utility is strongly class-dependent

The strongest trajectory benefit occurs for:

**`constraint_error`**

with a smaller positive effect for:

**`workflow_error`**

Trajectory context is neutral or harmful for several other failure families.

This family dependence explains why global metrics show only small improvements.

---

### 3. Constraint errors are the clearest target for trajectory-aware correction

The transition expert produces a substantial net improvement for constraint errors:

**+19 net rescues**

and approximately:

**+6 percentage points recall.**

The most promising corrections occur when the semantic expert predicts `workflow_error` or `tool_use_error`, but trajectory evidence suggests `constraint_error`.

This identifies a concrete failure mode where historical context matters.

---

### 4. The transition expert appears to compress the useful trajectory signal

Raw trajectory features are useful for constructing the transition representation, but they provide little additional routing information after the transition model has produced its posterior.

Trajectory-only override selection performs near chance, while posterior-based features perform substantially better.

Therefore the useful trajectory information appears to be **mediated through the transition expert's class probabilities**, rather than directly exploitable by a second-stage router.

---

### 5. Further selector engineering is unlikely to be the highest-value next step

Several increasingly sophisticated selection approaches were evaluated:

- global expert routing,
- compact interpretable routing,
- family-specific rules,
- thresholded constraint corrections,
- source-conditioned corrections,
- cross-fitted targeted corrections,
- learned override selection,
- trajectory-feature selection,
- and probability-space disagreement geometry.

All produce only modest improvements.

The best selector discrimination reached approximately:

- PR-AUC = **0.641**
- ROC-AUC = **0.706**

and downstream gains remained around one percentage point or less in OOF accuracy.

Given that the learned override dataset contains only **56 examples**, further gate complexity is likely to overfit rather than reveal substantially more signal.

---

# Final interpretation

The central result of this notebook is not that trajectory context should always be added.

Instead:

> **Trajectory context has selective utility. It is particularly valuable for failure types whose identity depends on interaction state rather than the current message alone, with `constraint_error` providing the clearest example.**

The semantic expert remains strong when the failure can be recognized directly from the current turn.

The transition expert becomes useful when classification requires information distributed across recent interaction history.

However, once trajectory information has been incorporated into the transition expert, the transition posterior itself captures most of the usable trajectory signal. Additional trajectory-based routing features provide little benefit.

This suggests a hierarchical interpretation of the system:

**current-turn semantics → baseline failure recognition**

**trajectory context → state-sensitive correction**

**expert posterior → compressed evidence of whether history changes the interpretation**

The remaining research question is therefore no longer simply:

> "Does trajectory context help?"

The notebook provides evidence that it does, selectively.

The more important next question is:

> **What properties of interaction history make trajectory context specifically useful for constraint errors?**

This motivates the next stage of analysis: a mechanism-level study of rescued `constraint_error` examples, including the prior interaction states, actions, constraints, and transitions that allow the trajectory expert to recover failures missed by current-turn semantics alone.